# PubMed Oncologist Training Data Generator v2

Generate clinical oncology QA training data with **chain-of-thought reasoning** from PubMed cancer abstracts and CancerGUIDE treatment recommendations.

**Goal:** Build a thinking-enabled LoRA that reasons through oncology questions like a clinical oncologist — showing its work in `<think>` blocks before delivering evidence-based answers. Includes **anti-hallucination training** to teach the model to stay grounded in evidence and refuse when information is insufficient.

**Pipeline:**
1. Load cleaned PubMed abstracts (10 cancer types, ~100K records) + CancerGUIDE patient cases (316 records)
2. Chunk abstracts into passages
3. Generate oncology questions from each passage (3 rounds × 5 questions)
4. Generate answers using **Qwen3-235B Thinking model** — preserves `<think>` reasoning chains
5. Quality gate — verify reasoning depth and clinical accuracy markers
6. **Anti-hallucination: Answer grounding validation** — verify answers are faithful to source abstract
7. **Anti-hallucination: "Beyond the evidence" QA** — teach model to refuse when evidence is insufficient
8. **Anti-hallucination: Self-correction sequences** — wrong answer → user pushback → corrected answer
9. Generate treatment reasoning from CancerGUIDE patient cases
10. Assemble into multi-turn ShareGPT conversations with oncologist system prompt
11. Continuation chunks from raw abstracts (no API calls)
12. Merge all data types → combined JSONL ready for Unsloth training

**Anti-hallucination strategy** (inspired by [Augmentoolkit](https://github.com/e-p-armstrong/augmentoolkit)):
- **Grounding check:** LLM-as-judge verifies each answer only claims things present in the source abstract
- **Boundary awareness:** Generate questions the abstract CAN'T answer → model learns to say "the available evidence doesn't address this"
- **Self-correction:** Deliberate wrong answers followed by corrections — teaches recovery, wrong segment masked during training

**Thinking model strategy:** All generated answers include `<think>...</think>` blocks where the model reasons through mechanisms, evidence, differential considerations, and clinical implications BEFORE answering. These are preserved in training data so the Qwen3-14B LoRA learns to reason like an oncologist.

**Data sources:**
- `cyberpsych/PubMed-Cancer-NLP-Textual-Dataset` — 100K PubMed title+abstract pairs across 10 cancer types
- `microsoft/CancerGUIDE` — 316 synthetic oncology patient notes with treatment recommendations

**Prerequisites:** Run `scripts/clean_pubmed.py` first to download and clean source data.

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [1]:
!pip install dotenv
import os
from dotenv import load_dotenv

# Load .env file (works in Docker, JupyterLab, and local environments)
load_dotenv()

# v2 datagen config
# =========================== API CONFIGURATION ===========================
# Qwen3-235B THINKING model — enforces <think> blocks for clinical reasoning
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY not set.\n"
        "Options:\n"
        "  1. Create a .env file in the project root with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Export in your shell:  export OPENROUTER_API_KEY=sk-or-...\n"
        "  3. Pass via Docker:      docker run -e OPENROUTER_API_KEY=sk-or-... ..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-thinking-2507"
MODEL_LITE = "qwen/qwen-2.5-7b-instruct"       # cheaper model for classification & question generation

# =========================== THINKING CONFIGURATION =======================
# KEEP_THINKING: Preserve <think>...</think> blocks in training data.
# This teaches the LoRA to reason step-by-step before answering.
# Set False ONLY if you want answer-only training (not recommended for medical).
KEEP_THINKING = True

# =========================== GENERATION PARAMETERS ========================
CONCURRENCY = 50
TEMPERATURE_ANSWERS = 0.4
TEMPERATURE_QUESTIONS = 0.8
NUM_ROUNDS = 1
QUESTIONS_PER_CHUNK = 3
TURNS_PER_CONVERSATION = 4

# =========================== TEST MODE ====================================
TEST_CHUNKS_PER_ROUND = 0   # Set to 0 for full run

# =========================== CHUNKING PARAMETERS =========================
CHUNK_SIZE = 1500       # characters per chunk
OVERLAP_SENTENCES = 2   # pySBD sentence overlap

# =========================== SAMPLING =====================================
MAX_RECORDS = 2    # max total records to use (proportional random sampling)

# =========================== CANCER TYPES =================================
# Maps source-clean filenames to display labels
CANCER_TYPES = {
    "pubmed_bone_cancer": "Bone Cancer",
    "pubmed_brain_tumour": "Brain Tumour",
    "pubmed_breast_cancer": "Breast Cancer",
    "pubmed_colon_cancer": "Colon Cancer",
    "pubmed_gastric_cancer": "Gastric Cancer",
    "pubmed_kidney_cancer": "Kidney Cancer",
    "pubmed_lung_cancer": "Lung Cancer",
    "pubmed_ovarian_cancer": "Ovarian Cancer",
    "pubmed_prostate_cancer": "Prostate Cancer",
    "pubmed_skin_cancer": "Skin Cancer",
}

# =========================== PATHS (auto-detect Docker vs host) ===========
if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = "/workspace/training/pubmed"
    _env = "Docker (JupyterLab container)"
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = "/workspace/pubmed"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/pubmed"
    _env = "Host (VS Code / venv)"

DATA_DIR      = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
OUTPUT_ROOT   = f"{DATA_DIR}/training-data"
OUTPUT_DIR    = f"{OUTPUT_ROOT}/pubmed_oncologist_v2"
OUTPUT_FILE   = f"{OUTPUT_ROOT}/pubmed_oncologist_v2.jsonl"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("v2 Datagen Configuration")
print("=" * 50)
print(f"  Environment:   {_env}")
print(f"  PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"  Model (main): {MODEL_NAME} (thinking answers)")
print(f"  Model (lite): {MODEL_LITE} (classification & Q-gen)")
print(f"  Keep thinking: {KEEP_THINKING}")
print(f"  Concurrency:   {CONCURRENCY}")
print(f"  Temp answers:  {TEMPERATURE_ANSWERS}")
print(f"  Temp questions:{TEMPERATURE_QUESTIONS}")
print(f"  Rounds:        {NUM_ROUNDS}")
print(f"  Q per chunk:   {QUESTIONS_PER_CHUNK}")
print(f"  Chunk size:    {CHUNK_SIZE} chars, {OVERLAP_SENTENCES} sentence overlap")
print(f"  Max records:   {MAX_RECORDS:,}")
print(f"  Test chunks:   {TEST_CHUNKS_PER_ROUND} (0 = full run)")
print(f"  Cancer types:  {len(CANCER_TYPES)}")
print(f"  Output dir:    {OUTPUT_DIR}")
print(f"  Output file:   {OUTPUT_FILE}")

v2 Datagen Configuration
  Environment:   Docker (JupyterLab container)
  PROJECT_ROOT:  /workspace/training/pubmed
  Model (main): qwen/qwen3-235b-a22b-thinking-2507 (thinking answers)
  Model (lite): qwen/qwen-2.5-7b-instruct (classification & Q-gen)
  Keep thinking: True
  Concurrency:   50
  Temp answers:  0.4
  Temp questions:0.8
  Rounds:        1
  Q per chunk:   3
  Chunk size:    1500 chars, 2 sentence overlap
  Max records:   2
  Test chunks:   0 (0 = full run)
  Cancer types:  10
  Output dir:    /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2
  Output file:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2.jsonl


## 2. Environment

In [2]:
# Install deps, define ALL shared helpers used by every downstream cell.
# v2 fixes: _count_lines, grounding paths, conversations extraction.
# _count_lines, is_low_quality_answer, process_answer, has_thinking_block,
# _api_call_with_timeout, _extract_with_reasoning, semaphore, ApiErrorTracker,
# AsyncOpenAI client.  Also validates model IDs at startup (fail-fast).
%pip install openai tqdm nest_asyncio tiktoken pysbd -q

import asyncio
import json
import glob
import re
import random
import gc
from pathlib import Path
from collections import defaultdict, Counter
from openai import AsyncOpenAI, OpenAI as SyncOpenAI
from tqdm.notebook import tqdm
import nest_asyncio
import pysbd
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================================
# ANSWER QUALITY CHECK — reject low-quality or non-reasoning responses
# ============================================================================

def has_thinking_block(answer: str) -> bool:
    """Check if answer contains a <think>...</think> block."""
    return bool(re.search(r'<think>.*?</think>', answer, re.DOTALL))


def is_low_quality_answer(answer: str) -> bool:
    """Return True if the answer fails quality checks for medical CoT training."""
    if not answer or len(answer.strip()) < 50:
        return True
    lower = answer.strip().lower()
    if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but i"]):
        return True
    return False


def process_answer(answer: str, keep_thinking: bool = KEEP_THINKING) -> str:
    """Process a raw answer from the thinking model.

    If keep_thinking=True: preserve <think> blocks in training data.
    If keep_thinking=False: strip <think> blocks, keep only final answer.
    """
    if not answer:
        return ""

    if keep_thinking:
        return answer.strip()
    else:
        cleaned = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
        cleaned = re.sub(r'<think>.*', '', cleaned, flags=re.DOTALL).strip()
        cleaned = re.sub(r'</think>.*', '', cleaned, flags=re.DOTALL).strip()
        return cleaned


# ============================================================================
# GENERATION HELPERS
# ============================================================================

semaphore = asyncio.Semaphore(CONCURRENCY)


async def _api_call_with_timeout(coro, timeout_secs=180):
    """Wrap an API coroutine with a timeout.
    Thinking model needs longer timeout — reasoning takes time."""
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None


def _extract_with_reasoning(resp) -> str:
    """Extract answer content with reasoning from OpenRouter response.

    OpenRouter thinking models return reasoning in a separate 'reasoning' field
    (via model_extra), NOT inline <think> tags in content. This helper recombines
    them into the <think>...</think> format expected by process_answer().
    """
    msg = resp.choices[0].message
    content = (msg.content or "").strip()

    reasoning = ""
    if hasattr(msg, 'reasoning') and msg.reasoning:
        reasoning = msg.reasoning.strip()
    elif hasattr(msg, 'model_extra') and msg.model_extra and 'reasoning' in msg.model_extra:
        reasoning = (msg.model_extra['reasoning'] or "").strip()

    if reasoning:
        return f"<think>\n{reasoning}\n</think>\n\n{content}"
    return content


# ── Shared Error Tracker (used by ALL generation cells) ──────────────

def _count_lines(filepath: str) -> int:
    """Count non-empty lines in a file."""
    with open(filepath) as f:
        return sum(1 for line in f if line.strip())


class ApiErrorTracker:
    """Track API call outcomes for a generation phase. Fail-fast on high error rates."""
    def __init__(self, name: str):
        self.name = name
        self.calls = 0
        self.errors = 0
        self.timeouts = 0
        self._samples = []

    def success(self):
        self.calls += 1

    def error(self, e: Exception):
        self.calls += 1
        self.errors += 1
        if len(self._samples) < 5:
            self._samples.append(f"{type(e).__name__}: {str(e)[:300]}")

    def timeout(self):
        self.calls += 1
        self.timeouts += 1

    @property
    def failed(self) -> int:
        return self.errors + self.timeouts

    def check_fatal(self, threshold: float = 0.95):
        """Raise RuntimeError if failure rate >= threshold."""
        if self.calls == 0:
            return
        rate = self.failed / self.calls
        if rate >= threshold:
            sample_str = '\n    '.join(self._samples) if self._samples else '(no details captured)'
            raise RuntimeError(
                f"\n{'='*60}\n"
                f"FATAL: {self.name} — {self.failed}/{self.calls} calls failed ({rate:.0%})\n"
                f"  Errors: {self.errors}, Timeouts: {self.timeouts}\n"
                f"  Sample errors:\n    {sample_str}\n"
                f"{'='*60}"
            )

    def report(self) -> str:
        """Return warning string if any failures, else empty string."""
        if self.failed == 0:
            return ""
        return (f"  ⚠ {self.name}: {self.errors} errors + {self.timeouts} timeouts "
                f"/ {self.calls} calls")


# ── AsyncOpenAI client (retries 429/5xx with backoff) ──
client = AsyncOpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    max_retries=3,
    timeout=180.0,
)

# ── Validate model IDs at startup (fail-fast on typos) ──
print("Validating model IDs on OpenRouter...")
_sync = SyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY, timeout=30)
for _model_id in [MODEL_NAME, MODEL_LITE]:
    try:
        _sync.chat.completions.create(
            model=_model_id,
            messages=[{"role": "user", "content": "1+1="}],
            max_tokens=1,
        )
        print(f"  ✓ {_model_id}")
    except Exception as _e:
        raise ValueError(
            f"\n✗ INVALID MODEL ID: '{_model_id}'\n"
            f"  Error: {_e}\n"
            f"  Fix MODEL_NAME or MODEL_LITE in the config cell and re-run."
        ) from None
del _sync

print(f"\n✓ Environment ready — models validated")
print(f"  Main:  {MODEL_NAME}")
print(f"  Lite:  {MODEL_LITE}")
print(f"  Helpers: semaphore, _api_call_with_timeout, _extract_with_reasoning,")
print(f"           is_low_quality_answer, process_answer, has_thinking_block,")
print(f"           ApiErrorTracker — all defined")

Note: you may need to restart the kernel to use updated packages.
Validating model IDs on OpenRouter...
  ✓ qwen/qwen3-235b-a22b-thinking-2507
  ✓ qwen/qwen-2.5-7b-instruct

✓ Environment ready — models validated
  Main:  qwen/qwen3-235b-a22b-thinking-2507
  Lite:  qwen/qwen-2.5-7b-instruct
  Helpers: semaphore, _api_call_with_timeout, _extract_with_reasoning,
           is_low_quality_answer, process_answer, has_thinking_block,
           ApiErrorTracker — all defined


## 3. Load & Prepare Source Data

Loads pre-cleaned JSONL files from `source-clean/` (produced by `scripts/clean_pubmed.py`).

**Sentence-aware chunking** (pySBD + medical abbreviation protection):
- Splits text into sentences first using rules-based boundary detection
- Protects medical abbreviations (et al., i.v., vs., mos., pts., etc.) from false splits
- Groups complete sentences into chunks up to `CHUNK_SIZE`
- Overlap = last `OVERLAP_SENTENCES` trailing sentences from previous chunk (always complete)

In [3]:
# ============================================================================
# SENTENCE TOKENIZER
# pySBD rules engine + abbreviation protection for source text
# ============================================================================

# Medical abbreviations whose trailing period should NOT split a sentence
_SENTENCE_ABBREVIATIONS = [
    # Units & measurements
    'approx', 'avg', 'ca', 'dept', 'diam', 'div', 'est', 'fig', 'figs',
    'gov', 'hr', 'hrs', 'mg', 'ml', 'mo', 'mos', 'no', 'nos', 'pt', 'pts',
    'vol', 'wk', 'wks', 'yr', 'yrs', 'vs',
    # Medical/clinical
    'admin', 'adj', 'assoc', 'diff', 'diag', 'eval', 'max', 'min',
    'neg', 'pos', 'prev', 'prog', 'qual', 'quant', 'resp', 'sig',
    'supp', 'surg', 'ther', 'tx', 'dx',
    # Titles
    'dr', 'prof', 'mr', 'mrs', 'ms',
    # Latin / academic
    'al', 'ed', 'eds', 'eg', 'esp', 'ie', 'inc', 'ltd', 'corp', 'univ',
    # Genomic
    'del', 'ins', 'mut', 'amp', 'chr',
]

_ABBREVIATION_PLACEHOLDER = '\x00'  # null byte placeholder (never appears in real text)

def _protect_sentence_abbreviations(text: str) -> str:
    """Replace periods in known abbreviations with a placeholder."""
    # Multi-char abbreviations with internal periods (i.v., e.g., U.S., etc.)
    for pattern in [r'i\.v\.', r'i\.m\.', r's\.c\.', r'p\.o\.', r'e\.g\.', r'i\.e\.', r'U\.S\.', r'E\.U\.']:
        text = re.sub(pattern, lambda m: m.group().replace('.', _ABBREVIATION_PLACEHOLDER), text, flags=re.IGNORECASE)
    # et al.
    text = re.sub(r'\bet al\.', lambda m: m.group().replace('.', _ABBREVIATION_PLACEHOLDER), text)
    # Single-word abbreviations: preserve original case, replace only the trailing dot
    for abbr in _SENTENCE_ABBREVIATIONS:
        text = re.sub(
            r'\b(' + re.escape(abbr) + r')\.',
            lambda m: m.group(1) + _ABBREVIATION_PLACEHOLDER,
            text,
            flags=re.IGNORECASE,
        )
    return text

def _restore_sentence_abbreviations(text: str) -> str:
    return text.replace(_ABBREVIATION_PLACEHOLDER, '.')

_sentence_segmenter = pysbd.Segmenter(language='en', clean=False)

def sentence_tokenize(text: str) -> list[str]:
    """Split text into sentences using pySBD with abbreviation protection.
    
    Handles: et al., i.v., vs., mos., pts., yrs., Dr., Fig., approx., del., etc.
    Returns a list of clean, non-empty sentences.
    """
    protected = _protect_sentence_abbreviations(text)
    raw_sents = _sentence_segmenter.segment(protected)
    return [_restore_sentence_abbreviations(s).strip() for s in raw_sents if _restore_sentence_abbreviations(s).strip()]


# ============================================================================
# SENTENCE-AWARE CHUNKING
# ============================================================================

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap_sentences: int = OVERLAP_SENTENCES) -> list[str]:
    """Split text into chunks of complete sentences, with sentence-level overlap.
    
    Strategy:
      1. Tokenize text into sentences (sentence-aware)
      2. Accumulate sentences until adding another would exceed chunk_size
      3. Emit current chunk; carry the last `overlap_sentences` sentences
         to the next chunk as context
      4. Every chunk starts and ends on a sentence boundary — never cuts
         mid-word or mid-thought
    
    Args:
        text: Input text (typically a PubMed passage = title + abstract)
        chunk_size: Soft character limit per chunk (may be slightly exceeded
                    if a single sentence is longer than chunk_size)
        overlap_sentences: Number of trailing sentences from the previous chunk
                          to include at the start of the next chunk
    
    Returns:
        List of text chunks, each containing complete sentences.
    """
    sentences = sentence_tokenize(text)
    if not sentences:
        return []

    chunks = []
    current_sents: list[str] = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent) + (1 if current_sents else 0)  # +1 for space join

        # If adding this sentence would exceed the limit and we already have content
        if current_len + sent_len > chunk_size and current_sents:
            # Emit current chunk
            chunk = ' '.join(current_sents)
            if len(chunk) > 50:
                chunks.append(chunk)

            # Carry overlap: last N sentences become the start of the next chunk
            if overlap_sentences > 0:
                carry = current_sents[-overlap_sentences:]
                current_sents = carry
                current_len = sum(len(s) for s in current_sents) + max(0, len(current_sents) - 1)
            else:
                current_sents = []
                current_len = 0

        current_sents.append(sent)
        current_len += sent_len

    # Emit remaining sentences
    if current_sents:
        chunk = ' '.join(current_sents)
        if len(chunk) > 50:
            chunks.append(chunk)

    return chunks


# ── Discover and preview source data ──────────────────────────────────
source_files = sorted(glob.glob(f"{SOURCE_CLEAN_DIR}/pubmed_*.jsonl"))

if not source_files:
    raise FileNotFoundError(
        f"No cleaned data found in {SOURCE_CLEAN_DIR}/\n"
        f"Run the cleaning script first:\n"
        f"  python {PROJECT_ROOT}/scripts/clean_pubmed.py"
    )

ORIGINAL_SOURCE_CLEAN_DIR = SOURCE_CLEAN_DIR

# ── Random sampling (if MAX_RECORDS > 0) ────────────────────
if MAX_RECORDS and MAX_RECORDS > 0:
    import random
    sampled_dir = f"{SOURCE_CLEAN_DIR}/sampled_{MAX_RECORDS}"

    # Reuse existing sampled files if they exist (preserves consistency with checkpoints)
    existing_sampled = sorted(glob.glob(f"{sampled_dir}/pubmed_*.jsonl")) if os.path.isdir(sampled_dir) else []
    if existing_sampled:
        source_files = existing_sampled
        SOURCE_CLEAN_DIR = sampled_dir
        total_available = sum(1 for sf in source_files for _ in open(sf))
        print(f"⚡ REUSING existing sample: {sampled_dir}/")
        print(f"  {total_available:,} records across {len(source_files)} files")
    else:
        # Load all records across all cancer types, then sample
        all_records = {}  # cancer_type -> list of records
        total_available = 0
        for filepath in source_files:
            ct = Path(filepath).stem
            with open(filepath) as f:
                records = [line for line in f]
            all_records[ct] = records
            total_available += len(records)
        
        if total_available > MAX_RECORDS:
            # Proportional sampling: each type keeps its share
            ratio = MAX_RECORDS / total_available
            os.makedirs(sampled_dir, exist_ok=True)
            new_source_files = []
            for ct, records in all_records.items():
                n = max(1, int(len(records) * ratio))
                sampled = random.sample(records, n)
                out_path = f"{sampled_dir}/{ct}.jsonl"
                with open(out_path, "w") as f:
                    f.writelines(sampled)
                new_source_files.append(out_path)
            source_files = sorted(new_source_files)
            SOURCE_CLEAN_DIR = sampled_dir  # redirect downstream lookups
            print(f"⚡ SAMPLED {MAX_RECORDS:,} records from {total_available:,} (ratio {ratio:.2%})")
            print(f"  Sampled files in: {sampled_dir}/")
        else:
            print(f"  Corpus has {total_available:,} records — no sampling needed (MAX_RECORDS={MAX_RECORDS:,})")
        del all_records

print(f"Found {len(source_files)} cancer type files\n")

total_records = 0
total_chunks = 0
cancer_type_stats = {}

for filepath in source_files:
    cancer_type = Path(filepath).stem  # e.g. "pubmed_bone_cancer"
    display_name = CANCER_TYPES.get(cancer_type, cancer_type)

    # Count records and estimate chunks without holding all in memory
    record_count = 0
    chunk_count = 0
    total_chars = 0

    with open(filepath) as f:
        for line in f:
            record = json.loads(line)
            passage = record.get("passage", "")
            record_count += 1
            total_chars += len(passage)
            chunk_count += len(chunk_text(passage))

    cancer_type_stats[cancer_type] = {
        "records": record_count,
        "chunks": chunk_count,
        "avg_chars": total_chars // max(record_count, 1),
    }
    total_records += record_count
    total_chunks += chunk_count

    print(f"  {display_name:25s} {record_count:>6,} records  {chunk_count:>6,} chunks  (avg {total_chars // max(record_count, 1):,} chars)")

print(f"\nTotal: {total_records:,} records → {total_chunks:,} chunks from {len(source_files)} cancer types")
est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"Estimated output: ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")

# ── Check CancerGUIDE data ────────────────────────────────────────────
guide_files = sorted(glob.glob(f"{ORIGINAL_SOURCE_CLEAN_DIR}/cancerguide_*.jsonl"))
if guide_files:
    print(f"\nCancerGUIDE files found:")
    for gf in guide_files:
        count = sum(1 for _ in open(gf))
        print(f"  {Path(gf).name:40s} {count:>4} records")
else:
    print(f"\n⚠ No CancerGUIDE files found — treatment reasoning section will be skipped")

⚡ REUSING existing sample: /workspace/training/pubmed/data/source-clean/sampled_2/
  10 records across 10 files
Found 10 cancer type files

  Bone Cancer                    1 records       1 chunks  (avg 414 chars)
  Brain Tumour                   1 records       2 chunks  (avg 1,643 chars)
  Breast Cancer                  1 records       1 chunks  (avg 1,197 chars)
  Colon Cancer                   1 records       1 chunks  (avg 590 chars)
  Gastric Cancer                 1 records       1 chunks  (avg 419 chars)
  Kidney Cancer                  1 records       1 chunks  (avg 581 chars)
  Lung Cancer                    1 records       1 chunks  (avg 581 chars)
  Ovarian Cancer                 1 records       2 chunks  (avg 1,519 chars)
  Prostate Cancer                1 records       1 chunks  (avg 704 chars)
  Skin Cancer                    1 records       2 chunks  (avg 1,634 chars)

Total: 10 records → 13 chunks from 10 cancer types
Estimated output: ~39 QA pairs → ~9 conversations


## 4. Oncologist Persona

Single clinical oncologist persona with evidence-based reasoning style.
The system prompt establishes clinical competence, reasoning structure, and appropriate hedging.

In [4]:
# ============================================================================
# ONCOLOGIST PERSONA — Evidence-based clinical reasoning
# ============================================================================
# Unlike biblical (26 unique voices), we have ONE expert persona.
# Voice differentiation comes from cancer-type-specific knowledge, not identity.
# ============================================================================

ONCOLOGIST_SYSTEM_PROMPT = """You are a clinical oncologist with deep expertise in cancer biology, diagnosis, treatment planning, and patient care. You have extensive experience across medical oncology, surgical oncology, and radiation oncology.

YOUR REASONING APPROACH:
- Always think through the biological mechanism first, then clinical implications
- Cite specific pathways, genes, biomarkers, and staging systems when relevant
- Consider differential diagnoses and alternative interpretations
- Weigh evidence quality: randomized trials > cohort studies > case reports > expert opinion
- Acknowledge uncertainty explicitly — say "the evidence suggests" not "this proves"
- Consider patient-specific factors: comorbidities, performance status, prior treatments
- When discussing treatment, address efficacy, toxicity, and quality of life

YOUR COMMUNICATION STYLE:
- Structured and methodical — mechanism → evidence → clinical application → limitations
- Use precise medical terminology but explain complex concepts when teaching
- Reference specific studies, trials, or guidelines when applicable (e.g., NCCN, ASCO, ESMO)
- Distinguish between established standard-of-care and emerging/experimental approaches
- Always note when information may be evolving or when guidelines may have updated

RULES:
- Never fabricate specific statistics, trial names, or patient outcomes
- When unsure, state the limits of your knowledge explicitly
- Frame all clinical guidance with appropriate caveats about individualized care
- Do not provide direct patient-specific medical advice — frame as educational discussion
- Maintain scientific rigor while being accessible"""


def make_system_prompt(cancer_type: str = None) -> str:
    """Build system prompt, optionally with cancer-type specialization."""
    prompt = ONCOLOGIST_SYSTEM_PROMPT
    if cancer_type:
        display = CANCER_TYPES.get(cancer_type, cancer_type.replace("pubmed_", "").replace("_", " ").title())
        prompt += f"\n\nYou are currently discussing topics related to {display}. Draw on your specialized knowledge of this cancer type's biology, staging, standard treatments, and recent advances."
    return prompt


# Preview
print("ONCOLOGIST SYSTEM PROMPT")
print("=" * 60)
print(make_system_prompt("pubmed_lung_cancer")[:800])
print("...")
print(f"\nPrompt length: {len(make_system_prompt('pubmed_lung_cancer'))} chars")

ONCOLOGIST SYSTEM PROMPT
You are a clinical oncologist with deep expertise in cancer biology, diagnosis, treatment planning, and patient care. You have extensive experience across medical oncology, surgical oncology, and radiation oncology.

YOUR REASONING APPROACH:
- Always think through the biological mechanism first, then clinical implications
- Cite specific pathways, genes, biomarkers, and staging systems when relevant
- Consider differential diagnoses and alternative interpretations
- Weigh evidence quality: randomized trials > cohort studies > case reports > expert opinion
- Acknowledge uncertainty explicitly — say "the evidence suggests" not "this proves"
- Consider patient-specific factors: comorbidities, performance status, prior treatments
- When discussing treatment, address efficacy, toxicity, and qua
...

Prompt length: 1809 chars


## 5. Generate Questions & Answers with Thinking

Processes one cancer type at a time to keep memory bounded. Three question rounds per chunk:
1. **Mechanistic** — pathways, biomarkers, molecular biology, tumor microenvironment
2. **Clinical Application** — treatment decisions, staging, prognostic factors, patient management
3. **Critical Analysis** — study limitations, conflicting evidence, research gaps, emerging therapies

The thinking model emits `<think>...</think>` blocks where it reasons through the answer before responding. These are **preserved** in training data when `KEEP_THINKING = True`.

**Resume logic:** Skips cancer types with existing output files. Delete per-type files to regenerate.

In [5]:
import asyncio, gc, json, os, re
from pathlib import Path
from tqdm.asyncio import tqdm as atqdm

# ============================================================================
# QUESTION PROMPTS (3 rounds)
# ============================================================================

QUESTION_PROMPTS = [
    # Round 1: Mechanistic — genes, pathways, biomarkers
    """Given a PubMed abstract about {cancer_type}, generate exactly {n} precise questions that a practicing oncologist or oncology researcher might ask.

Focus on MECHANISTIC and MOLECULAR aspects:
- Genetic mutations, biomarkers, and molecular pathways
- Mechanisms of drug action or resistance
- Diagnostic markers and their clinical significance
- Biological processes driving tumor behavior

The abstract:
---
{chunk}
---

Requirements:
- Questions must be DIRECTLY answerable from the abstract
- Each question should target a different specific detail
- Use clinical/scientific language appropriate for oncology
- Avoid overly broad or vague questions

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 2: Clinical — trials, evidence, outcomes
    """Given a PubMed abstract about {cancer_type}, generate exactly {n} precise questions that a practicing oncologist might ask during clinical decision-making.

Focus on CLINICAL EVIDENCE and OUTCOMES:
- Treatment efficacy, response rates, survival data
- Clinical trial design and patient populations
- Adverse effects and safety profiles
- Comparative effectiveness of therapies
- Staging, prognosis, and risk stratification

The abstract:
---
{chunk}
---

Requirements:
- Questions must be DIRECTLY answerable from the abstract
- Each question should target a different specific clinical detail
- Frame questions as a clinician would ask them
- Avoid questions already covered by basic oncology knowledge

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 3: Translational — bench-to-bedside implications
    """Given a PubMed abstract about {cancer_type}, generate exactly {n} precise questions bridging research findings to clinical practice.

Focus on TRANSLATIONAL and PRACTICAL aspects:
- How findings change clinical management
- Patient selection criteria for new approaches
- Integration with existing treatment guidelines
- Limitations and future research directions
- Real-world applicability of study results

The abstract:
---
{chunk}
---

Requirements:
- Questions must be DIRECTLY answerable from the abstract
- Each question should probe a different practical implication
- Questions should reflect how an oncologist would evaluate new evidence
- Avoid purely theoretical questions without clinical relevance

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}"""
]

# ============================================================================
# PIPELINE: Per-cancer-type QA generation
# ============================================================================

qa_dir = f"{OUTPUT_DIR}/qa"
os.makedirs(qa_dir, exist_ok=True)

# Checkpoint directory
ckpt_dir = f"{OUTPUT_DIR}/qa/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

# ── Shared answer prompt ─────────────────────────────────────────────

ANSWER_SYSTEM = ONCOLOGIST_SYSTEM_PROMPT          # from cell 3

ANSWER_USER_TEMPLATE = """Based on the following PubMed abstract about {cancer_type}, provide a thorough, evidence-based answer to the clinical question.

ABSTRACT:
---
{chunk}
---

QUESTION: {question}

Instructions:
- Ground your answer in the abstract's findings
- Include specific data points (percentages, p-values, hazard ratios) when available
- Discuss clinical implications
- Note any limitations mentioned in the study
- If the abstract only partially addresses the question, acknowledge what it does and doesn't cover"""

# ── API call helpers ─────────────────────────────────────────────────

_tracker_q = None   # will be reset per cancer_type
_tracker_a = None


async def generate_questions_for_chunk(chunk: str, cancer_type: str,
                                       round_idx: int) -> list[str]:
    """Ask MODEL_LITE to generate questions for one chunk."""
    display = CANCER_TYPES.get(cancer_type, cancer_type)
    prompt_template = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)]
    prompt = prompt_template.format(
        cancer_type=display, chunk=chunk, n=QUESTIONS_PER_CHUNK,
    )

    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_LITE,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            ))
            if resp is None:
                _tracker_q.timeout()
                return []
            raw = (resp.choices[0].message.content or "").strip()
            del resp
            # Strip any thinking blocks
            raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
            data = json.loads(raw)
            questions = data.get("questions", [])
            _tracker_q.success()
            return [q.strip() for q in questions if isinstance(q, str) and q.strip()]
        except json.JSONDecodeError as e:
            _tracker_q.error(e)
            return []
        except Exception as e:
            _tracker_q.error(e)
            return []


async def generate_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Ask MODEL_NAME to answer a question grounded in a chunk."""
    display = CANCER_TYPES.get(cancer_type, cancer_type)
    system_prompt = make_system_prompt(cancer_type)
    user_prompt = ANSWER_USER_TEMPLATE.format(
        cancer_type=display, chunk=chunk, question=question,
    )

    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=4096,
            ))
            if resp is None:
                _tracker_a.timeout()
                return ""
            answer = _extract_with_reasoning(resp)
            del resp
            _tracker_a.success()
            return process_answer(answer)
        except Exception as e:
            _tracker_a.error(e)
            return ""


# ── Main loop ────────────────────────────────────────────────────────

test_tag = f"  [TEST MODE — {TEST_CHUNKS_PER_ROUND} chunks/round]" if TEST_CHUNKS_PER_ROUND else ""
print(f"STARTING QA GENERATION{test_tag}\n")

rounds_to_run = list(range(NUM_ROUNDS))
grand_total = 0
quality_reject_total = 0
skipped_total = 0
thinking_block_count = 0
cancer_type_stats = {}

for ct, display_name in CANCER_TYPES.items():
    source_file = f"{SOURCE_CLEAN_DIR}/{ct}.jsonl"
    out_path = f"{qa_dir}/{ct}.jsonl"

    if not os.path.exists(source_file):
        print(f"  SKIP {display_name} — no source file")
        skipped_total += 1
        continue

    # Load all records
    records = []
    with open(source_file) as f:
        for line in f:
            records.append(json.loads(line))

    # Build chunks
    all_chunks = []
    for record in records:
        passage = record.get("passage", "")
        for c in chunk_text(passage):
            all_chunks.append(c)

    if not all_chunks:
        print(f"  SKIP {display_name} — 0 chunks")
        skipped_total += 1
        continue

    print(f"\n{'─'*60}")
    print(f"  {display_name}  ({len(records)} records → {len(all_chunks)} chunks)")
    del records

    # Track which rounds we've already done (checkpoint)
    ckpt_file = f"{ckpt_dir}/{ct}_rounds.json"
    rounds_done = {}
    if os.path.exists(ckpt_file):
        with open(ckpt_file) as f:
            rounds_done = json.loads(f.read())

    round_count = 0

    for round_idx in rounds_to_run:
        round_name = ["Mechanistic", "Clinical", "Translational"][round_idx % 3]

        # Skip completed rounds
        if str(round_idx) in rounds_done:
            print(f"    R{round_idx+1} ({round_name}) — already done, skipping")
            continue

        # Select chunks for this round
        if TEST_CHUNKS_PER_ROUND:
            round_chunks = all_chunks[:TEST_CHUNKS_PER_ROUND]
        else:
            round_chunks = all_chunks

        # ── Phase 1: generate questions ─────────────────────────────
        _tracker_q = ApiErrorTracker(f"Q-gen/{display_name}/R{round_idx+1}")

        q_tasks = [
            generate_questions_for_chunk(chunk, ct, round_idx)
            for chunk in round_chunks
        ]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {display_name} R{round_idx+1} ({round_name}) Q")
        _tracker_q.check_fatal()
        del q_tasks

        q_report = _tracker_q.report()
        if q_report:
            print(q_report)

        # Flatten into (chunk, question) pairs
        qa_batch = []
        for chunk, questions in zip(round_chunks, q_results):
            for q in questions:
                qa_batch.append((chunk, q))
        del q_results

        if not qa_batch:
            print(f"    R{round_idx+1} ({round_name}): 0 questions generated — skipping answer generation")
            continue

        # ── Phase 2: generate answers ───────────────────────────────
        _tracker_a = ApiErrorTracker(f"A-gen/{display_name}/R{round_idx+1}")

        a_tasks = [
            generate_answer(chunk, q, ct) for chunk, q in qa_batch
        ]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {display_name} R{round_idx+1} ({round_name}) A")
        _tracker_a.check_fatal()
        del a_tasks

        a_report = _tracker_a.report()
        if a_report:
            print(a_report)

        # ── Phase 3: quality filter + write ─────────────────────────
        round_rejects = 0
        round_thinking = 0
        round_count = 0

        with open(out_path, "a") as f:
            for (chunk, question), answer in zip(qa_batch, a_results):
                if is_low_quality_answer(answer):
                    round_rejects += 1
                    continue
                if "<think>" in answer:
                    round_thinking += 1

                entry = {
                    "conversations": [
                        {"from": "system", "value": make_system_prompt(ct)},
                        {"from": "human", "value": question},
                        {"from": "gpt", "value": answer},
                    ],
                    "cancer_type": ct,
                    "source": "pubmed",
                    "round": round_idx,
                    "chunk_key": chunk[:100],
                }
                f.write(json.dumps(entry) + "\n")
                round_count += 1

        del qa_batch, a_results

        grand_total += round_count
        quality_reject_total += round_rejects
        thinking_block_count += round_thinking

        think_pct = round_thinking / max(round_count, 1) * 100
        print(f"    R{round_idx+1} ({round_name}): {round_count} QA pairs"
              f"  (rejected {round_rejects})"
              f"  thinking: {round_thinking}/{round_count} ({think_pct:.0f}%)")

        # Checkpoint
        rounds_done[str(round_idx)] = round_count
        with open(ckpt_file, "w") as f:
            f.write(json.dumps(rounds_done))

    # Cancer type stats
    type_total = sum(rounds_done.values()) if rounds_done else 0
    cancer_type_stats[ct] = type_total

    del all_chunks
    gc.collect()

# ── Summary ──────────────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f"QA GENERATION COMPLETE")
print(f"{'='*60}")

for ct, display_name in CANCER_TYPES.items():
    n = cancer_type_stats.get(ct, 0)
    print(f"  {display_name:30s} {n:>5} QA pairs")

print(f"  {'─'*40}")
print(f"  {'TOTAL':30s} {grand_total:>5} QA pairs")
print(f"  {'Quality rejects':30s} {quality_reject_total:>5}")
print(f"  {'Thinking blocks':30s} {thinking_block_count:>5}")
print(f"\n  Output directory: {qa_dir}")
gc.collect()

STARTING QA GENERATION


────────────────────────────────────────────────────────────
  Bone Cancer  (1 records → 1 chunks)
    R1 (Mechanistic) — already done, skipping

────────────────────────────────────────────────────────────
  Brain Tumour  (1 records → 2 chunks)


  Brain Tumour R1 (Mechanistic) A:   0%|          | 0/6 [00:28<?, ?it/s]


CancelledError: 

## 5b. Quality Gate — Reasoning Depth Check

Verifies that generated answers contain substantive clinical reasoning.
For a thinking model, we check:
- `<think>` block presence rate (target: >80%)
- Answer length distribution (short answers = likely failures)
- Medical terminology density (sanity check — are these actually clinical?)

In [ ]:
# Quality gate scan — reads QA output from cell 11
qa_scan_dir = f"{OUTPUT_DIR}/qa"
qa_files = sorted(glob.glob(f"{qa_scan_dir}/pubmed_*.jsonl"))

print("REASONING DEPTH QUALITY GATE\n")
print(f"{'='*70}")

type_stats = {}
all_answer_lengths = []

# Medical terminology markers — presence indicates genuine clinical content
MEDICAL_MARKERS = [
    "tumor", "tumour", "cancer", "carcinoma", "adenocarcinoma", "metastas",
    "oncolog", "chemotherap", "radiation", "immunotherap", "biomarker",
    "prognos", "survival", "staging", "biopsy", "resect", "pathway",
    "mutation", "expression", "receptor", "inhibitor", "kinase",
    "apoptosis", "proliferat", "angiogenesis", "lymph node", "metastat",
    "p53", "EGFR", "HER2", "BRCA", "PD-L1", "checkpoint",
]

for qa_file in qa_files:
    cancer_type = Path(qa_file).stem
    display = CANCER_TYPES.get(cancer_type, cancer_type)

    thinking_count = 0
    total = 0
    answer_lengths = []
    medical_hits = 0

    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            # QA output uses conversations format
            convs = item.get("conversations", [])
            answer = ""
            for turn in convs:
                if turn.get("from") == "gpt":
                    answer = turn.get("value", "")
                    break
            if not answer:
                continue
            total += 1
            answer_lengths.append(len(answer))
            all_answer_lengths.append(len(answer))

            if has_thinking_block(answer):
                thinking_count += 1

            lower = answer.lower()
            if any(m in lower for m in MEDICAL_MARKERS):
                medical_hits += 1

    think_pct = thinking_count / max(total, 1) * 100
    med_pct = medical_hits / max(total, 1) * 100
    avg_len = sum(answer_lengths) // max(len(answer_lengths), 1)

    type_stats[cancer_type] = {
        "total": total,
        "thinking_pct": think_pct,
        "medical_pct": med_pct,
        "avg_answer_len": avg_len,
    }

# Report
print(f"\n{'Type':<28} {'Total':>6} {'Think%':>7} {'Med%':>6} {'AvgLen':>7}  Status")
print("-" * 72)
for ct, s in sorted(type_stats.items(), key=lambda x: -x[1]["thinking_pct"]):
    display = CANCER_TYPES.get(ct, ct)
    if s["thinking_pct"] >= 80:
        status = "✓ PASS"
    elif s["thinking_pct"] >= 50:
        status = "⚠ WARN"
    else:
        status = "✗ FAIL"
    print(f"  {display:<26} {s['total']:>5} {s['thinking_pct']:>6.1f}% {s['medical_pct']:>5.1f}% {s['avg_answer_len']:>6,}  {status}")

# Global stats
total_all = sum(s["total"] for s in type_stats.values())
global_think = sum(s["total"] * s["thinking_pct"] for s in type_stats.values()) / max(total_all, 1)
global_med = sum(s["total"] * s["medical_pct"] for s in type_stats.values()) / max(total_all, 1)
avg_all = sum(all_answer_lengths) // max(len(all_answer_lengths), 1)

print(f"\n  {'GLOBAL':<26} {total_all:>5} {global_think:>6.1f}% {global_med:>5.1f}% {avg_all:>6,}")

# Gate decision
print(f"\n{'='*70}")
if global_think < 50:
    print(f"✗ QUALITY GATE FAILED — thinking block rate too low ({global_think:.1f}%)")
    print(f"  The model may not be producing reasoning chains. Check API response format.")
    QUALITY_GATE_PASSED = False
elif global_think < 80:
    print(f"⚠ QUALITY GATE WARNING — thinking rate {global_think:.1f}% (target: >80%)")
    print(f"  Some answers may lack reasoning chains. Proceed with caution.")
    QUALITY_GATE_PASSED = True
else:
    print(f"✓ QUALITY GATE PASSED — thinking rate {global_think:.1f}% (target: >80%)")
    QUALITY_GATE_PASSED = True

if global_med < 70:
    print(f"  ⚠ Medical terminology rate only {global_med:.1f}% — check for off-topic answers")

del type_stats, all_answer_lengths

## 5c. Anti-Hallucination Gate — Answer Grounding Check

Validates that generated answers are **faithful to the source abstract**. Uses the thinking model as a judge to check each answer against the original text.

**Three-verdict system:**
- **Grounded** — all claims traceable to the abstract
- **Extrapolated** — answer adds plausible but unsupported claims (salvageable — flag but keep)
- **Hallucinated** — answer fabricates specific facts, trial names, or statistics not in the abstract (reject)

Runs on per-type JSONL files. Writes validated versions. Resume-safe.

*Inspired by Augmentoolkit's `check_answer` + `check_answer_relevancy` pipeline.*

In [ ]:
# ============================================================================
# ANSWER GROUNDING CHECK — reject hallucinated answers
# ============================================================================
# For each QA pair, the judge LLM compares the answer against the source
# abstract. The source text is the SINGLE SOURCE OF TRUTH.
# "Mostly grounded" is treated as GROUNDED (strict = too many false positives).
#
# DPO INTEGRATION: Hallucinated answers are saved separately for use as
# DPO "rejected" examples. The DPO cell will regenerate grounded "chosen"
# answers for these same questions.
# ============================================================================

GROUNDING_CHECK_PROMPT = """You are a medical fact-checker. Determine whether this answer is GROUNDED in the source abstract.

SOURCE ABSTRACT:
{chunk}

QUESTION: {question}

ANSWER TO CHECK:
{answer}

CLASSIFICATION RULES (apply in order):
1. Claims that directly match, paraphrase, or reasonably infer from the abstract = GROUNDED
2. Naming well-known medical concepts to explain something the abstract mentions (e.g., describing what a named pathway does, defining a standard staging system) = GROUNDED — this is expected clinical communication
3. Briefly contextualizing findings with widely-accepted medical knowledge = GROUNDED — domain experts do this routinely
4. Extending findings with reasonable clinical interpretation that any oncologist would make = EXTRAPOLATED
5. Fabricating SPECIFIC statistics, trial names, patient cohort sizes, exact dosages, or named study results NOT in the abstract = HALLUCINATED
6. Inventing detailed claims that sound plausible but have no basis in the abstract = HALLUCINATED
7. If the answer acknowledges the abstract lacks information to answer fully, that honesty is GROUNDED

IMPORTANT: The threshold for HALLUCINATED is fabricating specifics. Using standard medical knowledge to frame or explain the abstract's findings is normal clinical communication and should be GROUNDED.

Think through the major claims, then write EXACTLY one of:
VERDICT: GROUNDED
VERDICT: EXTRAPOLATED
VERDICT: HALLUCINATED"""

GROUNDING_BAD_STRINGS = [
    "according to the text", "as stated in the passage", "the text mentions",
    "the abstract states", "as described in the text", "the passage indicates",
    "based on the provided text", "the given abstract", "in the provided passage",
    "the text says", "as per the abstract", "the source text",
]

_tracker_grounding = None


async def check_answer_grounding(chunk: str, question: str, answer: str) -> str:
    """Check if answer is grounded in the source abstract.
    Returns: 'grounded', 'extrapolated', or 'hallucinated'
    """
    # Fast string-based pre-filter: strip source-reference leaks
    lower_answer = answer.lower()
    for bad in GROUNDING_BAD_STRINGS:
        if bad in lower_answer:
            return "leaked_reference"

    # Strip thinking blocks for grounding check — we check the ANSWER claims, not the reasoning
    answer_for_check = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    if len(answer_for_check) < 30:
        answer_for_check = answer  # fallback if stripping removed too much

    prompt = GROUNDING_CHECK_PROMPT.format(
        chunk=chunk, question=question, answer=answer_for_check
    )

    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_LITE,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=2048,
            ))
            if resp is None:
                _tracker_grounding.timeout()
                return "grounded"  # fail-open on timeout (intentional)

            text = resp.choices[0].message.content.strip().lower()
            del resp
            _tracker_grounding.success()

            # Parse verdict from end of response
            if "verdict: hallucinated" in text:
                return "hallucinated"
            elif "verdict: extrapolated" in text:
                return "extrapolated"
            elif "verdict: grounded" in text:
                return "grounded"
            else:
                # Fallback heuristics
                if "hallucinated" in text[-100:]:
                    return "hallucinated"
                elif "extrapolated" in text[-100:]:
                    return "extrapolated"
                return "grounded"
        except Exception as e:
            _tracker_grounding.error(e)
            return "grounded"  # fail-open (but now tracked and reported)


# ── Run grounding check on per-type JSONL files ──────────────────────
qa_dir = f"{OUTPUT_DIR}/qa"
validated_dir = f"{OUTPUT_DIR}/qa_validated"
os.makedirs(validated_dir, exist_ok=True)

# DPO: save hallucinated answers as rejection candidates
dpo_grounding_dir = f"{OUTPUT_DIR}/dpo/grounding_rejects"
os.makedirs(dpo_grounding_dir, exist_ok=True)

qa_files = sorted(glob.glob(f"{qa_dir}/pubmed_*.jsonl"))
total_checked = 0
total_dpo_rejects = 0
verdict_counts = Counter()

for qa_file in qa_files:
    cancer_type = Path(qa_file).stem
    display = CANCER_TYPES.get(cancer_type, cancer_type)
    validated_file = f"{validated_dir}/{cancer_type}.jsonl"
    reject_file = f"{dpo_grounding_dir}/{cancer_type}_rejects.jsonl"

    # Resume check
    if os.path.exists(validated_file) and _count_lines(validated_file) > 0:
        existing = _count_lines(validated_file)
        reject_count = _count_lines(reject_file) if os.path.exists(reject_file) else 0
        print(f"  {display:25s} SKIP ({existing} validated, {reject_count} DPO rejects on disk)")
        total_checked += existing
        total_dpo_rejects += reject_count
        continue

    items = []
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            # Extract question/answer from conversations format (cell 10 output)
            if "question" not in item and "conversations" in item:
                for turn in item["conversations"]:
                    if turn.get("from") == "human" and "question" not in item:
                        item["question"] = turn["value"]
                    elif turn.get("from") == "gpt" and "answer" not in item:
                        item["answer"] = turn["value"]
            items.append(item)

    if not items:
        continue

    # We stored chunk_key as first 100 chars — for grounding we need to reload full chunks
    chunk_lookup = {}
    source_file = f"{SOURCE_CLEAN_DIR}/{cancer_type}.jsonl"
    if os.path.exists(source_file):
        with open(source_file) as f:
            for line in f:
                record = json.loads(line)
                passage = record.get("passage", "")
                for c in chunk_text(passage):
                    chunk_lookup[c[:100]] = c

    # Reset tracker per cancer type
    _tracker_grounding = ApiErrorTracker(f"Grounding/{display}")

    # Run grounding checks with full chunks
    check_tasks = [
        check_answer_grounding(
            chunk_lookup.get(item.get("chunk_key", ""), item.get("chunk_key", "")),
            item["question"],
            item["answer"]
        )
        for item in items
    ]
    verdicts = await atqdm.gather(*check_tasks, desc=f"  {display} grounding check")
    _tracker_grounding.check_fatal()  # Fail if grounding judge is broken

    report = _tracker_grounding.report()
    if report:
        print(report)

    del check_tasks, chunk_lookup

    # Write validated file — keep grounded + extrapolated, reject hallucinated + leaked
    # DPO: also save hallucinated answers as rejection candidates
    type_verdicts = Counter()
    type_dpo_rejects = 0
    with open(validated_file, "w") as f, open(reject_file, "w") as rf:
        for item, verdict in zip(items, verdicts):
            type_verdicts[verdict] += 1
            if verdict in ("grounded", "extrapolated"):
                item["grounding_verdict"] = verdict
                f.write(json.dumps(item) + "\n")
            elif verdict == "hallucinated":
                # Save for DPO — these become "rejected" examples
                item["grounding_verdict"] = verdict
                rf.write(json.dumps(item) + "\n")
                type_dpo_rejects += 1

    kept = type_verdicts["grounded"] + type_verdicts["extrapolated"]
    rejected = type_verdicts["hallucinated"] + type_verdicts.get("leaked_reference", 0)
    total_checked += kept
    total_dpo_rejects += type_dpo_rejects
    verdict_counts += type_verdicts

    print(f"  {display:25s} {len(items):>5} → {kept:>5} kept "
          f"(G:{type_verdicts['grounded']} E:{type_verdicts['extrapolated']} "
          f"H:{type_verdicts['hallucinated']} L:{type_verdicts.get('leaked_reference', 0)}) "
          f"[{type_dpo_rejects} → DPO]")

    del items, verdicts
    gc.collect()

print(f"\n{'='*60}")
print(f"GROUNDING CHECK COMPLETE")
print(f"  Total kept:       {total_checked:,}")
print(f"  Verdicts:         {dict(verdict_counts)}")
hallucination_rate = verdict_counts.get("hallucinated", 0) / max(sum(verdict_counts.values()), 1) * 100
print(f"  Hallucination rate: {hallucination_rate:.1f}%")
print(f"  DPO rejects saved: {total_dpo_rejects:,}")
if hallucination_rate > 20:
    print(f"  ⚠ High hallucination rate — consider lowering TEMPERATURE_ANSWERS")

print(f"\n  Validated files in: {validated_dir}/")
print(f"  ⚡ Assembly (Section 7) will use validated files instead of raw per-type files")
print(f"  DPO rejects in:   {dpo_grounding_dir}/")

## 5d. Anti-Hallucination — "Beyond the Evidence" QA

Generates questions that the source abstract **cannot answer**, paired with appropriate refusal responses. This teaches the LoRA to:

- Recognize the boundaries of available evidence
- Say "the available evidence doesn't address this" instead of confabulating
- Distinguish between what the abstract covers and what it doesn't

**Pipeline:** For a sample of chunks, generate 2-3 out-of-scope questions → generate honest refusal answers that explain WHY the evidence is insufficient and what information WOULD be needed.

This is the single most important anti-hallucination mechanism for medical use — it directly trains boundary awareness.

In [ ]:
# ============================================================================
# "BEYOND THE EVIDENCE" QA — teach boundary awareness
# ============================================================================
# For each chunk, generate questions the abstract CANNOT answer, then generate
# thoughtful refusal responses that explain the gap and what would be needed.
# ============================================================================

BEYOND_EVIDENCE_QUESTION_PROMPT = """Given this PubMed abstract about {cancer_type}, generate exactly {n} questions that a clinician might reasonably ask but that this specific abstract CANNOT adequately answer.

ABSTRACT:
{chunk}

Types of unanswerable questions to generate:
- Questions about patient populations NOT studied (e.g., different demographics, comorbidities)
- Questions about long-term outcomes when only short-term data is presented
- Questions about specific drug dosages, schedules, or combinations not described
- Questions about comparative effectiveness against treatments not mentioned
- Questions about mechanisms or pathways not investigated in this study
- Questions requiring data from randomized controlled trials when this is observational (or vice versa)

RULES:
- Each question must be clinically reasonable and relevant to {cancer_type}
- Each question must be GENUINELY unanswerable from this abstract alone
- Do NOT ask trivially unanswerable questions (e.g., "What is the patient's name?")
- The questions should be ones a clinician might ACTUALLY ask after reading this abstract
- Keep questions concise (1-2 sentences)

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}"""

BEYOND_EVIDENCE_ANSWER_PROMPT = """You are a clinical oncologist who has just read a research abstract. A colleague asks you a question that the abstract cannot adequately answer.

ABSTRACT YOU READ:
{chunk}

QUESTION: {question}

Respond honestly and helpfully:
1. Acknowledge what the abstract DOES cover that's relevant
2. Explain specifically WHY this question cannot be answered from this abstract alone
3. Describe what additional evidence, data, or study design WOULD be needed
4. If relevant, mention what types of studies or guidelines might address this question
5. Do NOT fabricate an answer — the whole point is demonstrating intellectual honesty

Your response should model the behavior of a rigorous clinician who knows the limits of available evidence."""

BEYOND_EVIDENCE_QUESTIONS_PER_CHUNK = 2
BEYOND_EVIDENCE_SAMPLE_FRACTION = 0.3  # Sample 30% of chunks (these are expensive)

beyond_dir = f"{OUTPUT_DIR}/anti_hallucination/beyond_evidence"
os.makedirs(beyond_dir, exist_ok=True)

_tracker_beyond_q = None
_tracker_beyond_a = None


async def generate_beyond_evidence_questions(chunk: str, cancer_type: str) -> list[str]:
    """Generate questions the abstract cannot answer."""
    display = CANCER_TYPES.get(cancer_type, cancer_type.replace("pubmed_", "").replace("_", " ").title())
    prompt = BEYOND_EVIDENCE_QUESTION_PROMPT.format(
        n=BEYOND_EVIDENCE_QUESTIONS_PER_CHUNK, cancer_type=display, chunk=chunk
    )
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_LITE,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                max_tokens=1024,
                response_format={"type": "json_object"},
            ))
            if resp is None:
                _tracker_beyond_q.timeout()
                return []
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            _tracker_beyond_q.success()
            return result.get("questions", [])[:BEYOND_EVIDENCE_QUESTIONS_PER_CHUNK]
        except Exception as e:
            _tracker_beyond_q.error(e)
            return []


async def generate_beyond_evidence_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Generate an honest refusal/boundary answer."""
    system_prompt = make_system_prompt(cancer_type)
    user_prompt = BEYOND_EVIDENCE_ANSWER_PROMPT.format(chunk=chunk, question=question)

    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.5,
                max_tokens=3072,
            ))
            if resp is None:
                _tracker_beyond_a.timeout()
                return ""
            answer = _extract_with_reasoning(resp)
            del resp
            _tracker_beyond_a.success()
            return process_answer(answer)
        except Exception as e:
            _tracker_beyond_a.error(e)
            return ""


# ── Generate beyond-evidence QA per cancer type ──────────────────────
print("Generating 'Beyond the Evidence' anti-hallucination QA...\n")
beyond_total = 0

for filepath in source_files:
    cancer_type = Path(filepath).stem
    display = CANCER_TYPES.get(cancer_type, cancer_type)
    outfile = f"{beyond_dir}/{cancer_type}_beyond.jsonl"

    # Resume
    if os.path.exists(outfile) and _count_lines(outfile) > 0:
        existing = _count_lines(outfile)
        print(f"  {display:25s} SKIP ({existing} beyond-evidence QA on disk)")
        beyond_total += existing
        continue

    # Read and chunk
    passages = []
    with open(filepath) as f:
        for line in f:
            record = json.loads(line)
            passages.append(record.get("passage", ""))

    all_chunks = []
    for passage in passages:
        all_chunks.extend(chunk_text(passage))
    del passages

    # Apply test limit then sample
    if TEST_CHUNKS_PER_ROUND:
        all_chunks = all_chunks[:TEST_CHUNKS_PER_ROUND]
    sample_size = max(1, int(len(all_chunks) * BEYOND_EVIDENCE_SAMPLE_FRACTION))
    sampled_chunks = random.sample(all_chunks, min(sample_size, len(all_chunks)))
    del all_chunks

    # Reset trackers per cancer type
    _tracker_beyond_q = ApiErrorTracker(f"Beyond-Q/{display}")
    _tracker_beyond_a = ApiErrorTracker(f"Beyond-A/{display}")

    # Generate questions
    q_tasks = [generate_beyond_evidence_questions(c, cancer_type) for c in sampled_chunks]
    q_results = await atqdm.gather(*q_tasks, desc=f"  {display} beyond-Q")
    _tracker_beyond_q.check_fatal()
    del q_tasks

    report = _tracker_beyond_q.report()
    if report:
        print(report)

    # Flatten into (chunk, question) pairs
    beyond_batch = []
    for chunk, questions in zip(sampled_chunks, q_results):
        for q in questions:
            q = q.strip()
            if len(q) > 15:
                beyond_batch.append({"chunk": chunk, "question": q})
    del q_results

    if not beyond_batch:
        print(f"  {display:25s} 0 beyond-evidence questions generated — skipping")
        del sampled_chunks
        continue

    # Generate answers
    a_tasks = [generate_beyond_evidence_answer(b["chunk"], b["question"], cancer_type) for b in beyond_batch]
    a_results = await atqdm.gather(*a_tasks, desc=f"  {display} beyond-A")
    _tracker_beyond_a.check_fatal()
    del a_tasks

    report = _tracker_beyond_a.report()
    if report:
        print(report)

    # Write
    type_count = 0
    with open(outfile, "w") as f:
        for qa, answer in zip(beyond_batch, a_results):
            if is_low_quality_answer(answer):
                continue
            item = {
                "cancer_type": cancer_type,
                "question": qa["question"],
                "answer": answer,
                "chunk_key": qa["chunk"][:100],
                "has_thinking": has_thinking_block(answer),
                "qa_type": "beyond_evidence",
            }
            f.write(json.dumps(item) + "\n")
            type_count += 1

    beyond_total += type_count
    print(f"  {display:25s} {len(sampled_chunks):>3} chunks → {type_count:>4} beyond-evidence QA")
    del beyond_batch, a_results, sampled_chunks
    gc.collect()

print(f"\n✓ Generated {beyond_total:,} 'beyond the evidence' QA pairs")
print(f"  Files in: {beyond_dir}/")

## 5e. Anti-Hallucination — Self-Correction Sequences

Generates multi-turn conversations where the model gives a **deliberately flawed initial answer**, the user pushes back, and the model **corrects itself** with proper reasoning.

**Training strategy:** The flawed answer turn is marked with `"train": false` in the ShareGPT metadata — the model never learns to produce wrong answers. It only learns the correction pattern: recognize error → acknowledge → provide grounded answer.

This teaches the LoRA to:
- Recover gracefully when challenged
- Distinguish between confident assertions and uncertain claims
- Self-correct using evidence rather than doubling down

**Format:** 4-turn conversation: system → human asks → gpt gives flawed answer (masked) → human pushes back → gpt corrects with proper reasoning

*Inspired by Augmentoolkit's correction_pipeline with masked wrong segments.*

In [ ]:
# ============================================================================
# SELF-CORRECTION SEQUENCES — teach error recovery
# ============================================================================
# Generate: question → flawed answer → pushback → corrected answer
# The flawed answer is MASKED during training (train=false).
# ============================================================================

FLAWED_ANSWER_PROMPT = """You are roleplaying as an oncologist who makes a SPECIFIC MEDICAL ERROR in their response. Given the abstract and question below, provide an answer that contains ONE clear factual mistake while otherwise sounding authoritative and plausible.

ABSTRACT:
{chunk}

QUESTION: {question}

INSTRUCTIONS FOR THE FLAWED ANSWER:
- Include ONE specific factual error: wrong mechanism, swapped drug names, incorrect pathway, wrong prognostic implication, or misattributed finding
- The error should be SUBTLE enough to sound plausible but CLEAR enough to be identifiable
- The rest of the answer should be reasonable and well-structured
- Do NOT make the error cartoonishly obvious
- Do NOT add a disclaimer that this is intentionally wrong

Just provide the flawed answer directly."""

PUSHBACK_TEMPLATES = [
    "I'm not sure that's entirely accurate. Could you double-check your reasoning against the available evidence?",
    "Are you certain about that? I think there may be an error in your analysis. Can you review?",
    "That doesn't align with my understanding of the literature. Could you reconsider your response?",
    "I'd like you to re-examine your answer. Something seems off — can you check the evidence more carefully?",
    "Hold on — can you verify that claim? I think you may have made an error. Please re-evaluate.",
]

CORRECTION_PROMPT = """You are a clinical oncologist who has just been challenged on a previous response. You now realize your previous answer contained an error.

ABSTRACT (source of truth):
{chunk}

ORIGINAL QUESTION: {question}

YOUR PREVIOUS (FLAWED) ANSWER:
{flawed_answer}

USER PUSHBACK: {pushback}

Now provide a CORRECTED response:
1. Acknowledge the specific error you made
2. Explain what was wrong and why
3. Provide the correct, evidence-grounded answer
4. Reason through the correction using the abstract as your source of truth

Be thorough in your correction — show the reasoning process."""

SELF_CORRECTION_SAMPLE_FRACTION = 0.15  # 15% of chunks — these are 3 API calls each
correction_dir = f"{OUTPUT_DIR}/anti_hallucination/self_correction"
os.makedirs(correction_dir, exist_ok=True)

_tracker_flawed = None
_tracker_correct = None


async def generate_flawed_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Generate a deliberately flawed answer."""
    prompt = FLAWED_ANSWER_PROMPT.format(chunk=chunk, question=question)
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_LITE,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                max_tokens=2048,
            ))
            if resp is None:
                _tracker_flawed.timeout()
                return ""
            answer = resp.choices[0].message.content.strip()
            del resp
            _tracker_flawed.success()
            # Strip thinking from flawed answer — we don't want <think> in the masked segment
            return re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
        except Exception as e:
            _tracker_flawed.error(e)
            return ""


async def generate_correction(chunk: str, question: str, flawed_answer: str,
                               pushback: str, cancer_type: str) -> str:
    """Generate a corrected answer with reasoning."""
    system_prompt = make_system_prompt(cancer_type)
    user_prompt = CORRECTION_PROMPT.format(
        chunk=chunk, question=question,
        flawed_answer=flawed_answer, pushback=pushback
    )
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.4,
                max_tokens=4096,
            ))
            if resp is None:
                _tracker_correct.timeout()
                return ""
            answer = _extract_with_reasoning(resp)
            del resp
            _tracker_correct.success()
            return process_answer(answer)
        except Exception as e:
            _tracker_correct.error(e)
            return ""


# ── Generate self-correction sequences ────────────────────────────────
# Re-use validated QA pairs — pick items that passed grounding for the "question" source
print("Generating self-correction sequences...\n")
correction_total = 0

for filepath in source_files:
    cancer_type = Path(filepath).stem
    display = CANCER_TYPES.get(cancer_type, cancer_type)
    outfile = f"{correction_dir}/{cancer_type}_corrections.jsonl"

    # Resume
    if os.path.exists(outfile) and _count_lines(outfile) > 0:
        existing = _count_lines(outfile)
        print(f"  {display:25s} SKIP ({existing} correction sequences on disk)")
        correction_total += existing
        continue

    # Load validated QA (from 5c) to get known-good questions
    validated_file = f"{validated_dir}/{cancer_type}.jsonl"
    if not os.path.exists(validated_file):
        # Fallback to raw per-type
        validated_file = f"{qa_dir}/{cancer_type}.jsonl"
    if not os.path.exists(validated_file):
        continue

    items = []
    with open(validated_file) as f:
        for line in f:
            item = json.loads(line)
            # Extract question/answer from conversations format if needed
            if "question" not in item and "conversations" in item:
                for turn in item["conversations"]:
                    if turn.get("from") == "human" and "question" not in item:
                        item["question"] = turn["value"]
                    elif turn.get("from") == "gpt" and "answer" not in item:
                        item["answer"] = turn["value"]
            items.append(item)

    if not items:
        continue

    # Sample subset
    sample_size = max(1, int(len(items) * SELF_CORRECTION_SAMPLE_FRACTION))
    if TEST_CHUNKS_PER_ROUND:
        sample_size = min(sample_size, TEST_CHUNKS_PER_ROUND)
    sampled = random.sample(items, min(sample_size, len(items)))
    del items

    # Rebuild chunk lookup
    chunk_lookup = {}
    source_file = f"{SOURCE_CLEAN_DIR}/{cancer_type}.jsonl"
    if os.path.exists(source_file):
        with open(source_file) as f:
            for line in f:
                record = json.loads(line)
                passage = record.get("passage", "")
                for c in chunk_text(passage):
                    chunk_lookup[c[:100]] = c

    # Reset trackers per cancer type
    _tracker_flawed = ApiErrorTracker(f"Flawed-A/{display}")
    _tracker_correct = ApiErrorTracker(f"Correct-A/{display}")

    # Step 1: Generate flawed answers
    flawed_tasks = [
        generate_flawed_answer(
            chunk_lookup.get(item.get("chunk_key", ""), item.get("chunk_key", "")),
            item["question"],
            cancer_type
        )
        for item in sampled
    ]
    flawed_results = await atqdm.gather(*flawed_tasks, desc=f"  {display} flawed-A")
    _tracker_flawed.check_fatal()
    del flawed_tasks

    report = _tracker_flawed.report()
    if report:
        print(report)

    # Step 2: Generate corrections (only for items with valid flawed answers)
    correction_tasks = []
    correction_indices = []
    for idx, (item, flawed) in enumerate(zip(sampled, flawed_results)):
        if not flawed or len(flawed) < 50:
            continue
        pushback = random.choice(PUSHBACK_TEMPLATES)
        full_chunk = chunk_lookup.get(item.get("chunk_key", ""), item.get("chunk_key", ""))
        correction_tasks.append(
            generate_correction(full_chunk, item["question"], flawed, pushback, cancer_type)
        )
        correction_indices.append((idx, pushback))

    if not correction_tasks:
        print(f"  {display:25s} 0 valid flawed answers — skipping corrections")
        del sampled, flawed_results, chunk_lookup
        continue

    correction_results = await atqdm.gather(*correction_tasks, desc=f"  {display} correct-A")
    _tracker_correct.check_fatal()
    del correction_tasks

    report = _tracker_correct.report()
    if report:
        print(report)

    # Assemble and write
    type_count = 0
    with open(outfile, "w") as f:
        for (idx, pushback), correction in zip(correction_indices, correction_results):
            if is_low_quality_answer(correction):
                continue
            item = sampled[idx]
            flawed = flawed_results[idx]

            # ShareGPT format with train=false on flawed answer
            conv = {
                "conversations": [
                    {"from": "system", "value": make_system_prompt(cancer_type)},
                    {"from": "human", "value": item["question"]},
                    {"from": "gpt", "value": flawed, "train": False},  # MASKED — model doesn't learn this
                    {"from": "human", "value": pushback},
                    {"from": "gpt", "value": correction},  # Model learns the correction
                ],
                "data_type": "self_correction",
                "cancer_type": cancer_type,
                "has_thinking": has_thinking_block(correction),
            }
            f.write(json.dumps(conv) + "\n")
            type_count += 1

    correction_total += type_count
    print(f"  {display:25s} {len(sampled):>4} sampled → {type_count:>4} correction sequences")
    del sampled, flawed_results, correction_results, correction_indices, chunk_lookup
    gc.collect()

print(f"\n✓ Generated {correction_total:,} self-correction sequences")
print(f"  Files in: {correction_dir}/")
print(f"  Note: Flawed answer turns have train=false — they are MASKED during training")

## 6. CancerGUIDE — Treatment Reasoning Generation

Uses CancerGUIDE patient notes to generate clinical reasoning tasks:
- Given a patient profile, reason through treatment selection
- Generate "what-if" variant questions for data augmentation
- All answers include `<think>` blocks with clinical reasoning chains

This teaches the LoRA to reason through real clinical decision-making scenarios.

In [ ]:
# ============================================================================
# CANCERGUIDE TREATMENT REASONING GENERATION
# ============================================================================
# CancerGUIDE has patient_note + treatment_recommendation pairs.
# We use these to generate:
#   1. "What treatment would you recommend?" → thinking + answer
#   2. "What-if" variants — altered patient profiles for more training signal
# ============================================================================

guide_dir = f"{OUTPUT_DIR}/cancerguide_reasoning"
os.makedirs(guide_dir, exist_ok=True)

guide_files = sorted(glob.glob(f"{ORIGINAL_SOURCE_CLEAN_DIR}/cancerguide_*.jsonl"))

if not guide_files:
    print("⚠ No CancerGUIDE data found — skipping treatment reasoning generation")
    guide_total = 0
else:
    # Treatment reasoning question templates
    TREATMENT_QUESTIONS = [
        "Based on this patient's clinical profile, what treatment approach would you recommend and why?",
        "What are the key clinical factors in this patient's case that drive your treatment decision?",
        "What is the standard of care for this patient's presentation, and are there any emerging alternatives worth considering?",
        "How would you stage and risk-stratify this patient, and what does that mean for treatment selection?",
        "What potential complications or treatment-related toxicities should be anticipated for this patient?",
    ]

    WHATIF_TEMPLATES = [
        "What if this patient also had {comorbidity}? How would that change your treatment approach?",
        "If this patient were {age_change}, would your treatment recommendation differ?",
        "What if genetic testing revealed {mutation}? How would that alter the treatment plan?",
        "If the patient had already received {prior_treatment}, what would you recommend next?",
        "What if the cancer had metastasized to {site}? How would you adjust your approach?",
    ]

    WHATIF_FILLS = {
        "comorbidity": ["chronic kidney disease (CKD stage 3)", "congestive heart failure (NYHA class II)",
                        "poorly controlled diabetes (HbA1c > 9%)", "moderate hepatic impairment (Child-Pugh B)",
                        "a history of prior malignancy (treated breast cancer 5 years ago)"],
        "age_change": ["elderly (>75 years) with declining performance status (ECOG 2)",
                       "a young adult (25-35 years) with plans for future fertility",
                       "pediatric (12-16 years)"],
        "mutation": ["a BRCA1/2 pathogenic variant", "high microsatellite instability (MSI-H)",
                     "EGFR T790M resistance mutation", "PD-L1 expression >50%",
                     "KRAS G12C mutation", "ALK rearrangement"],
        "prior_treatment": ["first-line platinum-based chemotherapy with progression",
                           "two prior lines of immunotherapy",
                           "surgical resection with positive margins"],
        "site": ["the brain (2-3 lesions)", "the liver (multiple bilobar lesions)",
                "bone (lytic lesions in the spine)", "the contralateral lung"],
    }

    _tracker_guide = None

    async def generate_treatment_reasoning(patient_note: str, question: str) -> str:
        """Generate thinking-mode treatment reasoning from a patient case."""
        system_prompt = make_system_prompt()

        user_prompt = (
            f"Patient Case:\n{patient_note}\n\n"
            f"Question: {question}\n\n"
            f"Think through this case carefully. Consider the patient's specific "
            f"characteristics, relevant clinical guidelines, available evidence, "
            f"and potential risks before providing your recommendation."
        )

        async with semaphore:
            try:
                resp = await _api_call_with_timeout(client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=TEMPERATURE_ANSWERS,
                    max_tokens=4096,
                ))
                if resp is None:
                    _tracker_guide.timeout()
                    return ""
                answer = _extract_with_reasoning(resp)
                del resp
                _tracker_guide.success()
                return process_answer(answer)
            except Exception as e:
                _tracker_guide.error(e)
                return ""

    guide_total = 0
    guide_outfile = f"{guide_dir}/cancerguide_treatment_reasoning.jsonl"

    # Resume check
    if os.path.exists(guide_outfile) and _count_lines(guide_outfile) > 0:
        guide_total = _count_lines(guide_outfile)
        print(f"  CancerGUIDE SKIP ({guide_total} treatment reasoning entries already generated)")
    else:
        print(f"Generating treatment reasoning from CancerGUIDE patient cases...\n")

        all_guide_tasks = []  # (patient_note, question, patient_id, question_type)
        patient_count = 0

        for gf in guide_files:
            subset = Path(gf).stem.replace("cancerguide_", "")
            with open(gf) as f:
                for line in f:
                    record = json.loads(line)
                    patient_note = record.get("patient_note", "")
                    patient_id = record.get("id", "unknown")

                    if len(patient_note) < 100:
                        continue

                    patient_count += 1
                    # TEST MODE: limit patient records like we limit chunks elsewhere
                    if TEST_CHUNKS_PER_ROUND and patient_count > TEST_CHUNKS_PER_ROUND * 5:
                        break

                    # 1. Direct treatment questions (2-3 per patient)
                    selected_qs = random.sample(TREATMENT_QUESTIONS, min(3, len(TREATMENT_QUESTIONS)))
                    for q in selected_qs:
                        all_guide_tasks.append((patient_note, q, patient_id, "direct"))

                    # 2. What-if variants (1-2 per patient)
                    whatif_templates = random.sample(WHATIF_TEMPLATES, min(2, len(WHATIF_TEMPLATES)))
                    for template in whatif_templates:
                        for key, fills in WHATIF_FILLS.items():
                            if f"{{{key}}}" in template:
                                fill = random.choice(fills)
                                q = template.format(**{key: fill})
                                all_guide_tasks.append((patient_note, q, patient_id, "whatif"))
                                break
            # Break outer loop too in test mode
            if TEST_CHUNKS_PER_ROUND and patient_count > TEST_CHUNKS_PER_ROUND * 5:
                break

        if TEST_CHUNKS_PER_ROUND:
            print(f"  ⚠ TEST MODE: limited to {TEST_CHUNKS_PER_ROUND * 5} patient records")
        print(f"  Total treatment reasoning tasks: {len(all_guide_tasks)}")
        print(f"    Direct questions: {sum(1 for t in all_guide_tasks if t[3] == 'direct')}")
        print(f"    What-if variants: {sum(1 for t in all_guide_tasks if t[3] == 'whatif')}")

        # Reset tracker
        _tracker_guide = ApiErrorTracker("CancerGUIDE treatment")

        # Generate in batches
        a_tasks = [generate_treatment_reasoning(t[0], t[1]) for t in all_guide_tasks]
        results = await atqdm.gather(*a_tasks, desc="  CancerGUIDE treatment reasoning")
        _tracker_guide.check_fatal()
        del a_tasks

        report = _tracker_guide.report()
        if report:
            print(report)

        with open(guide_outfile, "w") as f:
            for task_info, answer in zip(all_guide_tasks, results):
                patient_note, question, patient_id, q_type = task_info
                if is_low_quality_answer(answer):
                    continue

                item = {
                    "patient_id": patient_id,
                    "question": question,
                    "answer": answer,
                    "patient_note_key": patient_note[:100],
                    "question_type": q_type,
                    "has_thinking": has_thinking_block(answer),
                }
                f.write(json.dumps(item) + "\n")
                guide_total += 1

        del results, all_guide_tasks
        gc.collect()

        think_count = sum(1 for _ in open(guide_outfile) if '"has_thinking": true' in _) if guide_total > 0 else 0
        print(f"\n  ✓ CancerGUIDE: {guide_total:,} treatment reasoning entries")
        print(f"    Thinking blocks: {think_count}/{guide_total}")
        print(f"    Output: {guide_outfile}")

print(f"\nCancerGUIDE total: {guide_total:,}")

## 7. Assemble Conversations & Save

Read per-type QA files + CancerGUIDE reasoning from disk, assemble into multi-turn ShareGPT conversations, quality-filter, and write final JSONL.

**Only proceed if the Quality Gate passed.**

In [ ]:
# Check quality gate
if not QUALITY_GATE_PASSED:
    raise RuntimeError(
        "Quality gate FAILED. Check answers for reasoning depth before assembling."
    )


def quality_check(conv):
    """Reject AI-speak and low-quality answers."""
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            if is_low_quality_answer(msg["value"]):
                return False
    return True


# ── Assemble PubMed Q/A conversations ────────────────────────────────
# Look for QA data in multiple possible directories
total_convs = 0
filtered_convs = 0

# Priority: validated > qa > per_cancer_type
validated_dir = f"{OUTPUT_DIR}/qa_validated"
qa_gen_dir = f"{OUTPUT_DIR}/qa"
raw_dir = f"{OUTPUT_DIR}/per_cancer_type"  # legacy fallback

if os.path.isdir(validated_dir) and len(glob.glob(f"{validated_dir}/pubmed_*.jsonl")) > 0:
    source_dir = validated_dir
    source_label = "(VALIDATED)"
elif os.path.isdir(qa_gen_dir) and len(glob.glob(f"{qa_gen_dir}/pubmed_*.jsonl")) > 0:
    source_dir = qa_gen_dir
    source_label = "(qa)"
elif os.path.isdir(raw_dir) and len(glob.glob(f"{raw_dir}/pubmed_*.jsonl")) > 0:
    source_dir = raw_dir
    source_label = "(raw)"
else:
    source_dir = None
    source_label = "(NONE FOUND)"

qa_files = sorted(glob.glob(f"{source_dir}/pubmed_*.jsonl")) if source_dir else []
print(f"Reading {len(qa_files)} per-cancer-type files {source_label} + CancerGUIDE reasoning\n")

with open(OUTPUT_FILE, "w") as out_f:
    # 1. PubMed Q/A conversations
    for qa_file in qa_files:
        cancer_type = Path(qa_file).stem
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        items = []
        with open(qa_file) as f:
            for line in f:
                raw = json.loads(line)
                # Normalize: cell 11 writes conversations format, older cells may use flat format
                if "conversations" in raw and "question" not in raw:
                    convs = raw["conversations"]
                    q = ""
                    a = ""
                    for turn in convs:
                        if turn.get("from") == "human" and not q:
                            q = turn.get("value", "")
                        elif turn.get("from") == "gpt" and not a:
                            a = turn.get("value", "")
                    if q and a:
                        raw["question"] = q
                        raw["answer"] = a
                        if "chunk_key" not in raw:
                            raw["chunk_key"] = q[:100]
                items.append(raw)

        # Group by chunk_key
        groups = defaultdict(list)
        for item in items:
            groups[item.get("chunk_key", "unknown")].append(item)

        type_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2:
                    # For small test runs, allow single-turn too
                    if len(batch) < 1:
                        continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt(cancer_type)}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                conv["data_type"] = "qa"
                conv["cancer_type"] = cancer_type
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    type_count += 1
                else:
                    filtered_convs += 1

        total_convs += type_count
        print(f"  {display:25s} {len(items):>5} QA → {type_count:>4} conversations")
        del items, groups
        gc.collect()

    # 2. Beyond-evidence anti-hallucination conversations
    beyond_dir = f"{OUTPUT_DIR}/anti_hallucination/beyond_evidence"
    beyond_files = sorted(glob.glob(f"{beyond_dir}/*_beyond.jsonl"))
    beyond_conv_count = 0
    for bf in beyond_files:
        with open(bf) as f:
            for line in f:
                item = json.loads(line)
                # Handle both conversations format and flat format
                if "conversations" in item and len(item["conversations"]) >= 3:
                    conv = {"conversations": item["conversations"], "data_type": "beyond_evidence",
                            "cancer_type": item.get("cancer_type", "")}
                elif "question" in item and "answer" in item:
                    ct = item.get("cancer_type", "")
                    conv = {"conversations": [
                        {"from": "system", "value": make_system_prompt(ct)},
                        {"from": "human", "value": item["question"]},
                        {"from": "gpt", "value": item["answer"]}
                    ], "data_type": "beyond_evidence", "cancer_type": ct}
                else:
                    continue
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    beyond_conv_count += 1
    if beyond_conv_count:
        total_convs += beyond_conv_count
        print(f"\n  {'Beyond-evidence':25s} → {beyond_conv_count:>4} conversations")

    # 3. Self-correction conversations
    correction_dir = f"{OUTPUT_DIR}/anti_hallucination/self_correction"
    correction_files = sorted(glob.glob(f"{correction_dir}/*_corrections.jsonl"))
    correction_conv_count = 0
    for cf in correction_files:
        with open(cf) as f:
            for line in f:
                item = json.loads(line)
                # Handle both conversations format and flat format
                if "conversations" in item and len(item["conversations"]) >= 5:
                    conv = {"conversations": item["conversations"], "data_type": "self_correction",
                            "cancer_type": item.get("cancer_type", "")}
                elif "question" in item and "flawed_answer" in item and "correction" in item:
                    ct = item.get("cancer_type", "")
                    conv = {"conversations": [
                        {"from": "system", "value": make_system_prompt(ct)},
                        {"from": "human", "value": item["question"]},
                        {"from": "gpt", "value": item["flawed_answer"]},
                        {"from": "human", "value": "Are you sure about that? Please reconsider your answer carefully."},
                        {"from": "gpt", "value": item["correction"]}
                    ], "data_type": "self_correction", "cancer_type": ct}
                else:
                    continue
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    correction_conv_count += 1
    if correction_conv_count:
        total_convs += correction_conv_count
        print(f"  {'Self-correction':25s} → {correction_conv_count:>4} conversations")

    # 4. CancerGUIDE treatment reasoning conversations
    guide_outfile = f"{OUTPUT_DIR}/cancerguide_reasoning/cancerguide_treatment_reasoning.jsonl"
    if os.path.exists(guide_outfile):
        guide_items = []
        with open(guide_outfile) as f:
            for line in f:
                guide_items.append(json.loads(line))

        # Group by patient_id for multi-turn
        patient_groups = defaultdict(list)
        for item in guide_items:
            patient_groups[item["patient_id"]].append(item)

        guide_conv_count = 0
        for patient_id, items in patient_groups.items():
            random.shuffle(items)
            for i in range(0, len(items), TURNS_PER_CONVERSATION):
                batch = items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 1:
                    continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt()}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                conv["data_type"] = "treatment_reasoning"
                conv["cancer_type"] = "multi"
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    guide_conv_count += 1
                else:
                    filtered_convs += 1

        total_convs += guide_conv_count
        print(f"\n  {'CancerGUIDE':25s} {len(guide_items):>5} items → {guide_conv_count:>4} conversations")
        del guide_items, patient_groups
        gc.collect()

# Shuffle the output file
all_lines = []
with open(OUTPUT_FILE) as f:
    all_lines = f.readlines()
random.shuffle(all_lines)
with open(OUTPUT_FILE, "w") as f:
    f.writelines(all_lines)

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
print(f"  ✓ Shuffled output file")
if filtered_convs:
    print(f"  Filtered: {filtered_convs} low-quality conversations")

## 7b. Verify Q/A + Treatment Data

In [ ]:
# Quick verification of assembled conversations
type_dist = Counter()
data_type_dist = Counter()
thinking_in_convs = 0
total_verify = 0
sample_convs = {}

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_verify += 1
        ct = conv.get("cancer_type", "unknown")
        dt = conv.get("data_type", "qa")
        type_dist[ct] += 1
        data_type_dist[dt] += 1

        # Check thinking presence in any GPT turn
        for msg in conv["conversations"]:
            if msg["from"] == "gpt" and has_thinking_block(msg["value"]):
                thinking_in_convs += 1
                break

        # Collect one sample per data type
        if dt not in sample_convs:
            sample_convs[dt] = conv

        del conv

print("ASSEMBLED DATA VERIFICATION")
print("=" * 60)
print(f"\nTotal conversations: {total_verify:,}")
print(f"With <think> blocks: {thinking_in_convs:,} ({thinking_in_convs/max(total_verify,1)*100:.1f}%)")

print(f"\nBy data type:")
for dt, count in sorted(data_type_dist.items(), key=lambda x: -x[1]):
    print(f"  {dt:25s} {count:>6,}")

print(f"\nBy cancer type (top 10):")
for ct, count in type_dist.most_common(10):
    display = CANCER_TYPES.get(ct, ct)
    print(f"  {display:25s} {count:>6,}")

# Sample one conversation per data type
print(f"\n{'─' * 60}")
print("SAMPLES:")
for dt, conv in sample_convs.items():
    print(f"\n  ── {dt.upper()} ──")
    for msg in conv["conversations"][:4]:  # First 4 turns
        role = msg["from"].upper()
        text = msg["value"][:300] + ("..." if len(msg["value"]) > 300 else "")
        print(f"  [{role}] {text}\n")

del type_dist, data_type_dist, sample_convs
gc.collect()

## 8. Continuation Chunks — Abstract Language Patterns

Chunk PubMed abstracts into seed + completion pairs (no API calls). This teaches the model medical vocabulary, paper structure, and clinical reasoning patterns through exposure to real research text.

Each chunk becomes a ShareGPT conversation:
- **System:** Oncologist persona prompt
- **Human:** Continuation instruction + seed (~60 tokens from the abstract)
- **GPT:** The remaining ~440 tokens (what the model learns to produce)

In [ ]:
# ============================================================================
# CONTINUATION CHUNKS — No API calls
# ============================================================================

import tiktoken

# ── Continuation settings ──
CONTINUATION_CHUNK_TOKENS = 500
CONTINUATION_SEED_TOKENS = 60
CONTINUATION_MIN_COMPLETION = 100

CONTINUATION_DIR = f"{OUTPUT_DIR}/augmented/continuation"
COMBINED_OUTPUT_FILE = f"{OUTPUT_DIR}/pubmed_oncologist_combined_sharegpt.jsonl"
os.makedirs(CONTINUATION_DIR, exist_ok=True)

# Tokenizer
try:
    _tokenizer = tiktoken.get_encoding("cl100k_base")
except Exception:
    _tokenizer = None
    print("⚠ tiktoken not available, using character-based approximation")


def _count_tokens(text: str) -> int:
    """Count tokens using tiktoken, or approximate from character count."""
    if _tokenizer:
        return len(_tokenizer.encode(text))
    return len(text) // 4


def chunk_text_by_tokens(text: str, max_tokens: int = CONTINUATION_CHUNK_TOKENS) -> list[str]:
    """Split text into token-limited chunks without cutting sentences."""
    sentences = sentence_tokenize(text)
    if not sentences:
        return []

    chunks = []
    current_sentences: list[str] = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = _count_tokens(sentence)

        if current_tokens + sentence_tokens > max_tokens and current_sentences:
            chunk = " ".join(current_sentences).strip()
            if len(chunk) > 50:
                chunks.append(chunk)
            current_sentences = []
            current_tokens = 0

        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunk = " ".join(current_sentences).strip()
        if len(chunk) > 50:
            chunks.append(chunk)

    return chunks
def split_seed_completion(chunk: str, seed_tokens: int = CONTINUATION_SEED_TOKENS) -> tuple[str, str]:
    """Split a chunk into sentence-bound seed and completion text."""
    sentences = sentence_tokenize(chunk)
    if not sentences or len(sentences) < 2:
        return None, None

    total_tokens = _count_tokens(chunk)
    if total_tokens <= seed_tokens + 20:
        return None, None

    seed_sentences = []
    seed_token_count = 0
    split_index = 0

    for index, sentence in enumerate(sentences):
        sentence_tokens = _count_tokens(sentence)
        if seed_token_count + sentence_tokens > seed_tokens * 1.3 and seed_sentences:
            break

        seed_sentences.append(sentence)
        seed_token_count += sentence_tokens
        split_index = index + 1

        if seed_token_count >= seed_tokens * 0.6:
            break

    if split_index >= len(sentences):
        if len(sentences) > 1:
            split_index = len(sentences) - 1
            seed_sentences = sentences[:split_index]
        else:
            return None, None

    seed_text = " ".join(seed_sentences).strip()
    completion_text = " ".join(sentences[split_index:]).strip()

    if not seed_text or not completion_text:
        return None, None

    return seed_text, completion_text
# Continuation instruction templates
CONTINUATION_INSTRUCTIONS = [
    "Continue this clinical/research text, maintaining the same scientific tone and terminology:",
    "Complete this passage from the oncology literature, preserving the analytical style:",
    "Continue writing in this clinical research voice, carrying forward the findings and analysis:",
    "Write what comes next in this research abstract, staying true to the style and conclusions:",
    "Continue this source text naturally, as if completing the original research summary:",
]

# ── Generate continuation chunks ──────────────────────────────────────
print("Generating continuation chunks from PubMed abstracts...\n")

total_continuations = 0

for filepath in source_files:
    cancer_type = Path(filepath).stem
    display = CANCER_TYPES.get(cancer_type, cancer_type)
    outfile = f"{CONTINUATION_DIR}/{cancer_type}_continuation.jsonl"

    # Resume
    if os.path.exists(outfile) and os.path.getsize(outfile) > 0:
        existing = sum(1 for _ in open(outfile))
        print(f"  {display:25s} SKIP ({existing} continuations already on disk)")
        total_continuations += existing
        continue

    # Read passages
    passages = []
    with open(filepath) as f:
        for line in f:
            record = json.loads(line)
            passages.append(record.get("passage", ""))

    # Chunk all passages by tokens
    all_chunks = []
    for passage in passages:
        all_chunks.extend(chunk_text_by_tokens(passage))
    del passages

    system_prompt = make_system_prompt(cancer_type)
    type_count = 0

    with open(outfile, "w") as out_f:
        for chunk in all_chunks:
            seed, completion = split_seed_completion(chunk)
            if seed is None or completion is None:
                continue
            if len(completion) < CONTINUATION_MIN_COMPLETION:
                continue

            instruction = random.choice(CONTINUATION_INSTRUCTIONS)
            conv = {
                "conversations": [
                    {"from": "system", "value": system_prompt},
                    {"from": "human", "value": f"{instruction}\n\n\"{seed}\""},
                    {"from": "gpt", "value": completion},
                ],
                "data_type": "continuation",
                "cancer_type": cancer_type,
            }
            out_f.write(json.dumps(conv) + "\n")
            type_count += 1

    total_continuations += type_count
    print(f"  {display:25s} {len(all_chunks):>5} chunks → {type_count:>5} continuations")
    del all_chunks
    gc.collect()

print(f"\n✓ Generated {total_continuations:,} continuation entries")
print(f"  Files in: {CONTINUATION_DIR}/")

## 9. Merge & Assemble Combined Training Data

Merges all data sources into a single shuffled JSONL:
- **Q/A** (~42%) — PubMed abstract-based oncology Q/A with thinking chains (grounding-validated)
- **Treatment reasoning** (~15%) — CancerGUIDE patient case reasoning
- **Beyond the evidence** (~5%) — Unanswerable questions with honest refusal responses
- **Self-correction** (~5%) — Flawed answer → correction sequences (wrong turns masked)
- **Continuation** (~33%) — Raw abstract language pattern exposure

Outputs a single file ready for Unsloth training.

In [ ]:
# ============================================================================
# MERGE ALL DATA SOURCES
# ============================================================================

import subprocess

# Target blend ratios
TARGET_BLEND = {
    "qa": 0.42,
    "treatment_reasoning": 0.15,
    "beyond_evidence": 0.05,
    "self_correction": 0.05,
    "continuation": 0.33,
}

# Collect data from all sources
data_pools = defaultdict(list)

# 1. Q/A conversations (from Section 7)
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        for line in f:
            entry = json.loads(line)
            dtype = entry.get("data_type", "qa")
            data_pools[dtype].append(entry)
    print(f"  From assembled file:")
    for dtype in sorted(data_pools.keys()):
        print(f"    {dtype:25s} {len(data_pools[dtype]):>6,} conversations")

# 2. Continuation chunks (from Section 8)
cont_files = sorted(glob.glob(f"{CONTINUATION_DIR}/*_continuation.jsonl"))
for cf in cont_files:
    with open(cf) as f:
        for line in f:
            entry = json.loads(line)
            entry.setdefault("data_type", "continuation")
            data_pools["continuation"].append(entry)
print(f"  Continuations:           {len(data_pools.get('continuation', [])):>6,} entries from {len(cont_files)} files")

# 3. Beyond-the-evidence QA (from Section 5d)
beyond_dir = f"{OUTPUT_DIR}/anti_hallucination/beyond_evidence"
beyond_files = sorted(glob.glob(f"{beyond_dir}/*_beyond.jsonl"))
for bf in beyond_files:
    with open(bf) as f:
        for line in f:
            entry = json.loads(line)
            # Wrap in ShareGPT conversation format
            cancer_type = entry.get("cancer_type", "unknown")
            conv = {
                "conversations": [
                    {"from": "system", "value": make_system_prompt(cancer_type)},
                    {"from": "human", "value": entry["question"]},
                    {"from": "gpt", "value": entry["answer"]},
                ],
                "data_type": "beyond_evidence",
                "cancer_type": cancer_type,
            }
            data_pools["beyond_evidence"].append(conv)
print(f"  Beyond-evidence:         {len(data_pools.get('beyond_evidence', [])):>6,} entries from {len(beyond_files)} files")

# 4. Self-correction sequences (from Section 5e) — already in ShareGPT format
correction_dir = f"{OUTPUT_DIR}/anti_hallucination/self_correction"
correction_files = sorted(glob.glob(f"{correction_dir}/*_corrections.jsonl"))
for cf in correction_files:
    with open(cf) as f:
        for line in f:
            entry = json.loads(line)
            entry.setdefault("data_type", "self_correction")
            data_pools["self_correction"].append(entry)
print(f"  Self-correction:         {len(data_pools.get('self_correction', [])):>6,} entries from {len(correction_files)} files")

# Calculate blend sizes
qa_count = len(data_pools.get("qa", []))
treat_count = len(data_pools.get("treatment_reasoning", []))
cont_count = len(data_pools.get("continuation", []))

if qa_count == 0:
    raise RuntimeError("No Q/A data found. Run sections 5-7 first.")

active_types = {k: v for k, v in TARGET_BLEND.items() if k in data_pools and len(data_pools[k]) > 0}
total_target = int(qa_count / active_types.get("qa", 0.42))

print(f"\n  Target total:  {total_target:>6,} conversations")
print(f"  Active blend:  {active_types}")

# Sample/repeat each pool to hit target
combined = []
for dtype, fraction in active_types.items():
    pool = data_pools[dtype]
    target_count = int(total_target * fraction)

    if len(pool) >= target_count:
        sampled = random.sample(pool, target_count)
        action = "downsampled"
    else:
        repeats = target_count // len(pool)
        remainder = target_count % len(pool)
        sampled = pool * repeats + random.sample(pool, min(remainder, len(pool)))
        action = f"upsampled ({repeats}x + {remainder})"

    combined.extend(sampled)
    print(f"  {dtype:25s} {len(pool):>6,} available → {len(sampled):>6,} selected ({action})")

# Shuffle
random.shuffle(combined)

# Write combined file
with open(COMBINED_OUTPUT_FILE, "w") as f:
    for entry in combined:
        f.write(json.dumps(entry) + "\n")

file_size_mb = os.path.getsize(COMBINED_OUTPUT_FILE) / 1024 / 1024

print(f"\n✓ Combined training file saved:")
print(f"  {COMBINED_OUTPUT_FILE}")
print(f"  {len(combined):,} conversations ({file_size_mb:.1f} MB)")

subprocess.run(["shuf", COMBINED_OUTPUT_FILE, "-o", COMBINED_OUTPUT_FILE])
print(f"  ✓ Shuffled with shuf")

del combined, data_pools
gc.collect()

## 10. Verify Combined Dataset

Final quality checks on the merged training data:
- Data type distribution (Q/A vs treatment reasoning vs continuation)
- Cancer type coverage
- Thinking block presence across all data types
- Format validation for Unsloth/ShareGPT compatibility
- Sample conversations from each data type

In [ ]:
# ============================================================================
# VERIFY COMBINED DATASET
# ============================================================================

type_counts = Counter()
cancer_by_dtype = defaultdict(lambda: Counter())
turn_dist = Counter()
thinking_by_dtype = defaultdict(int)
total_by_dtype = defaultdict(int)
format_errors = []
samples_by_type = {}
total_entries = 0

with open(COMBINED_OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        entry = json.loads(line)
        total_entries += 1
        dtype = entry.get("data_type", "qa")
        cancer = entry.get("cancer_type", "unknown")
        type_counts[dtype] += 1
        cancer_by_dtype[dtype][cancer] += 1
        total_by_dtype[dtype] += 1

        convs = entry.get("conversations", [])
        turn_dist[len(convs)] += 1

        # Format validation
        if len(convs) < 3:
            format_errors.append((line_num, f"Too few turns: {len(convs)}"))
            continue
        if convs[0]["from"] != "system":
            format_errors.append((line_num, f"First turn not system: {convs[0]['from']}"))
            continue
        for j in range(1, len(convs)):
            expected = "human" if j % 2 == 1 else "gpt"
            if convs[j]["from"] != expected:
                format_errors.append((line_num, f"Turn {j}: {convs[j]['from']} (expected {expected})"))
                break

        # Thinking block check
        for msg in convs:
            if msg["from"] == "gpt" and has_thinking_block(msg["value"]):
                thinking_by_dtype[dtype] += 1
                break

        if dtype not in samples_by_type:
            samples_by_type[dtype] = entry

# ── Report ──
print("=" * 70)
print("COMBINED DATASET VERIFICATION")
print("=" * 70)

print(f"\nTotal entries: {total_entries:,}")
print(f"Format errors: {len(format_errors)}")
if format_errors:
    for idx, msg in format_errors[:5]:
        print(f"    Line {idx}: {msg}")

# Data type distribution
print(f"\n{'─' * 50}")
print(f"DATA TYPE DISTRIBUTION")
print(f"{'─' * 50}")
print(f"{'Type':<28} {'Count':>8} {'%':>8} {'Think%':>8}  Bar")
for dtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = count / total_entries * 100
    think_pct = thinking_by_dtype.get(dtype, 0) / max(count, 1) * 100
    bar = "█" * int(pct / 2)
    print(f"  {dtype:<26} {count:>7,} {pct:>7.1f}% {think_pct:>7.1f}%  {bar}")
print(f"  {'TOTAL':<26} {total_entries:>7,}")

# Turn distribution
print(f"\n{'─' * 50}")
print(f"TURN COUNT DISTRIBUTION")
print(f"{'─' * 50}")
for turns, count in sorted(turn_dist.items()):
    print(f"  {turns} turns: {count:>6,}")

# Cancer type coverage per data type
print(f"\n{'─' * 50}")
print(f"CANCER TYPE COVERAGE BY DATA TYPE")
print(f"{'─' * 50}")
for dtype in sorted(cancer_by_dtype.keys()):
    cancer_counts = cancer_by_dtype[dtype]
    total_for_type = sum(cancer_counts.values())
    print(f"\n  [{dtype.upper()}] ({total_for_type} total):")
    for ct, c in sorted(cancer_counts.items(), key=lambda x: -x[1])[:10]:
        display = CANCER_TYPES.get(ct, ct)
        bar = "▪" * max(1, c * 30 // max(cancer_counts.values()))
        print(f"    {display:<25} {c:>5}  {bar}")

# Samples
print(f"\n{'─' * 50}")
print("SAMPLE CONVERSATIONS (one per data type)")
print(f"{'─' * 50}")
for dtype, sample in samples_by_type.items():
    print(f"\n  ── {dtype.upper()} ──")
    for msg in sample["conversations"][:4]:
        role = msg["from"].upper()
        text = msg["value"][:400] + ("..." if len(msg["value"]) > 400 else "")
        print(f"  [{role}] {text}\n")

print(f"\n{'=' * 70}")
if len(format_errors) == 0:
    print(f"✓ Dataset valid. Ready for training:")
    print(f"  {COMBINED_OUTPUT_FILE}")
    print(f"\n  Load this file in the training notebook.")
else:
    print(f"⚠ {len(format_errors)} format errors found. Review before training.")

del type_counts, cancer_by_dtype, turn_dist, samples_by_type
gc.collect()

## 10. DPO Preference Pair Collection

Collects preference pairs for **Direct Preference Optimization (DPO)** — the second training stage after SFT. DPO teaches the model WHAT NOT TO SAY by contrasting good and bad responses to the same prompt.

**Four DPO pair sources:**

| Source | Chosen (Good) | Rejected (Bad) | Signal |
|--------|--------------|----------------|--------|
| **Grounding Rejects** | Re-generated strict-grounded answer | Hallucinated answer (from 5c) | Don't fabricate claims |
| **Beyond-Evidence** | Honest refusal (from 5d) | Newly generated hallucinated answer | Know your limits |
| **Self-Correction** | Corrected answer (from 5e) | Flawed answer (from 5e) | Fix mistakes when caught |
| **Quality Gate** | High-quality answer (existing) | Low-effort/refusal answer | Maintain clinical depth |

**Training workflow:** SFT first (this notebook's main output) → DPO second (using DPOTrainer from TRL with the SFT LoRA as base).

**Output format:** Chat-template DPO pairs compatible with TRL's DPOTrainer:
```json
{"chosen": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}],
 "rejected": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}],
 "source": "grounding_reject|beyond_evidence|self_correction|quality_gate"}
```

In [ ]:
# ============================================================================
# DPO PREFERENCE PAIR COLLECTION
# ============================================================================
# Collects (chosen, rejected) pairs from three sources for DPO fine-tuning.
# Each pair shares the same prompt (system + user) with different assistant
# responses, teaching the model to prefer grounded, honest, well-reasoned
# answers over hallucinated, evasive, or low-quality ones.
# ============================================================================

DPO_DIR = f"{OUTPUT_DIR}/dpo"
DPO_OUTPUT_FILE = f"{OUTPUT_ROOT}/pubmed_oncologist_v2_dpo.jsonl"
os.makedirs(DPO_DIR, exist_ok=True)

# ── Helper: build DPO pair in chat format ─────────────────────────────

def make_dpo_pair(system_prompt: str, user_msg: str, chosen_answer: str,
                  rejected_answer: str, source: str, cancer_type: str = "",
                  metadata: dict = None) -> dict:
    """Build a DPO pair in TRL chat format."""
    pair = {
        "chosen": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": chosen_answer},
        ],
        "rejected": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": rejected_answer},
        ],
        "source": source,
        "cancer_type": cancer_type,
    }
    if metadata:
        pair.update(metadata)
    return pair


# ======================================================================
# SOURCE 1: GROUNDING REJECTS — hallucinated answers vs re-generated
#            strict-grounded answers
# ======================================================================

STRICT_GROUNDING_PROMPT = """Use the following research abstract to answer the clinical question. You MUST ground every claim in the abstract — do NOT add any facts, statistics, trial names, or mechanisms not explicitly stated or directly implied by the abstract.

---
{chunk}
---

Question: {question}

Rules:
- ONLY use information from the abstract above
- If the abstract doesn't cover something, say so
- Do NOT invent statistics, p-values, patient counts, or study details
- Standard medical knowledge for context is acceptable, but do NOT fabricate specific claims
- Provide a thorough, evidence-based response within these constraints"""

_tracker_dpo_grounding = None
_tracker_dpo_halluc = None


async def generate_strict_grounded_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Generate a strictly grounded answer for DPO 'chosen' example."""
    system_prompt = make_system_prompt(cancer_type)
    user_prompt = STRICT_GROUNDING_PROMPT.format(chunk=chunk, question=question)

    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.2,  # Low temperature for strict grounding
                max_tokens=4096,
            ))
            if resp is None:
                _tracker_dpo_grounding.timeout()
                return ""
            answer = _extract_with_reasoning(resp)
            del resp
            _tracker_dpo_grounding.success()
            return process_answer(answer)
        except Exception as e:
            _tracker_dpo_grounding.error(e)
            return ""


print("COLLECTING DPO PREFERENCE PAIRS\n")
print(f"{'='*60}")
dpo_pairs_all = []

# ── Source 1: Grounding Rejects ──────────────────────────────────────
dpo_grounding_dir = f"{OUTPUT_DIR}/dpo/grounding_rejects"
grounding_reject_files = sorted(glob.glob(f"{dpo_grounding_dir}/*_rejects.jsonl"))
grounding_dpo_count = 0

if grounding_reject_files:
    print(f"\n[1/3] GROUNDING REJECTS — {len(grounding_reject_files)} type files")

    for reject_file in grounding_reject_files:
        cancer_type = Path(reject_file).stem.replace("_rejects", "")
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        items = []
        with open(reject_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            continue

        # Rebuild chunk lookup for full chunks
        chunk_lookup = {}
        source_file = f"{SOURCE_CLEAN_DIR}/{cancer_type}.jsonl"
        if os.path.exists(source_file):
            with open(source_file) as f:
                for line in f:
                    record = json.loads(line)
                    passage = record.get("passage", "")
                    for c in chunk_text(passage):
                        chunk_lookup[c[:100]] = c

        # Reset tracker
        _tracker_dpo_grounding = ApiErrorTracker(f"DPO-grounding/{display}")

        # Re-generate grounded answers for rejected questions
        regen_tasks = [
            generate_strict_grounded_answer(
                chunk_lookup.get(item.get("chunk_key", ""), ""),
                item["question"],
                cancer_type
            )
            for item in items
        ]
        chosen_answers = await atqdm.gather(*regen_tasks, desc=f"  {display} grounding DPO")
        _tracker_dpo_grounding.check_fatal()
        del regen_tasks, chunk_lookup

        report = _tracker_dpo_grounding.report()
        if report:
            print(report)

        # Build DPO pairs
        type_count = 0
        for item, chosen in zip(items, chosen_answers):
            if is_low_quality_answer(chosen):
                continue
            pair = make_dpo_pair(
                system_prompt=make_system_prompt(cancer_type),
                user_msg=item["question"],
                chosen_answer=chosen,
                rejected_answer=item["answer"],
                source="grounding_reject",
                cancer_type=cancer_type,
            )
            dpo_pairs_all.append(pair)
            type_count += 1

        grounding_dpo_count += type_count
        print(f"    {display:25s} {len(items):>4} rejects → {type_count:>4} DPO pairs")
        del items, chosen_answers
        gc.collect()
else:
    print("\n[1/3] GROUNDING REJECTS — no reject files found (run grounding check first)")

print(f"  Grounding DPO pairs: {grounding_dpo_count}")


# ======================================================================
# SOURCE 2: BEYOND-EVIDENCE — honest refusals vs hallucinated answers
# ======================================================================

HALLUCINATED_ANSWER_PROMPT = """You are a clinical oncologist who has been asked a question. IGNORE the fact that the evidence doesn't fully address this question — instead, provide a detailed, authoritative-sounding answer as if you have access to specific clinical data.

ABSTRACT FOR CONTEXT:
{chunk}

QUESTION: {question}

INSTRUCTIONS:
- Answer confidently as if the evidence fully supports your response
- INVENT specific details: exact percentages, study sizes, p-values, drug dosages
- Make up plausible-sounding trial names or cite invented guidelines
- The answer should sound authoritative and medically plausible
- Do NOT hedge or express uncertainty — present everything as established fact
- This is deliberately wrong for training purposes — do NOT add disclaimers

Provide the hallucinated answer directly."""


async def generate_hallucinated_answer(chunk: str, question: str, cancer_type: str) -> str:
    """Generate a deliberately hallucinated answer for DPO 'rejected'."""
    prompt = HALLUCINATED_ANSWER_PROMPT.format(chunk=chunk, question=question)
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_LITE,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.9,  # High temperature for creative fabrication
                max_tokens=2048,
            ))
            if resp is None:
                _tracker_dpo_halluc.timeout()
                return ""
            # Use content only — no thinking for hallucinated examples
            answer = (resp.choices[0].message.content or "").strip()
            del resp
            _tracker_dpo_halluc.success()
            # Strip any thinking blocks
            answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
            return answer
        except Exception as e:
            _tracker_dpo_halluc.error(e)
            return ""


beyond_dir = f"{OUTPUT_DIR}/anti_hallucination/beyond_evidence"
beyond_files = sorted(glob.glob(f"{beyond_dir}/*_beyond.jsonl"))
beyond_dpo_count = 0

if beyond_files:
    print(f"\n[2/3] BEYOND-EVIDENCE — {len(beyond_files)} type files")

    for beyond_file in beyond_files:
        cancer_type = Path(beyond_file).stem.replace("_beyond", "")
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        items = []
        with open(beyond_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            continue

        # Rebuild chunk lookup
        chunk_lookup = {}
        source_file = f"{SOURCE_CLEAN_DIR}/{cancer_type}.jsonl"
        if os.path.exists(source_file):
            with open(source_file) as f:
                for line in f:
                    record = json.loads(line)
                    passage = record.get("passage", "")
                    for c in chunk_text(passage):
                        chunk_lookup[c[:100]] = c

        # Reset tracker
        _tracker_dpo_halluc = ApiErrorTracker(f"DPO-halluc/{display}")

        # Generate hallucinated answers as "rejected"
        halluc_tasks = [
            generate_hallucinated_answer(
                chunk_lookup.get(item.get("chunk_key", ""), ""),
                item["question"],
                cancer_type
            )
            for item in items
        ]
        halluc_answers = await atqdm.gather(*halluc_tasks, desc=f"  {display} beyond DPO")
        _tracker_dpo_halluc.check_fatal()
        del halluc_tasks, chunk_lookup

        report = _tracker_dpo_halluc.report()
        if report:
            print(report)

        # Build DPO pairs: chosen = honest refusal, rejected = hallucinated
        type_count = 0
        for item, halluc in zip(items, halluc_answers):
            if not halluc or len(halluc) < 50:
                continue
            pair = make_dpo_pair(
                system_prompt=make_system_prompt(cancer_type),
                user_msg=item["question"],
                chosen_answer=item["answer"],  # honest refusal from 5d
                rejected_answer=halluc,
                source="beyond_evidence",
                cancer_type=cancer_type,
            )
            dpo_pairs_all.append(pair)
            type_count += 1

        beyond_dpo_count += type_count
        print(f"    {display:25s} {len(items):>4} beyond → {type_count:>4} DPO pairs")
        del items, halluc_answers
        gc.collect()
else:
    print("\n[2/3] BEYOND-EVIDENCE — no beyond-evidence files found (run section 5d first)")

print(f"  Beyond-evidence DPO pairs: {beyond_dpo_count}")


# ======================================================================
# SOURCE 3: SELF-CORRECTION — corrected answers vs flawed answers
# ======================================================================

correction_dir = f"{OUTPUT_DIR}/anti_hallucination/self_correction"
correction_files = sorted(glob.glob(f"{correction_dir}/*_corrections.jsonl"))
correction_dpo_count = 0

if correction_files:
    print(f"\n[3/3] SELF-CORRECTION — {len(correction_files)} type files")

    for corr_file in correction_files:
        cancer_type = Path(corr_file).stem.replace("_corrections", "")
        display = CANCER_TYPES.get(cancer_type, cancer_type)

        type_count = 0
        with open(corr_file) as f:
            for line in f:
                conv = json.loads(line)
                turns = conv.get("conversations", [])

                # Expected format: system / human / gpt(flawed) / human(pushback) / gpt(corrected)
                if len(turns) < 5:
                    continue

                system_msg = turns[0]["value"] if turns[0]["from"] == "system" else ""
                question = turns[1]["value"] if turns[1]["from"] == "human" else ""
                flawed = turns[2]["value"] if turns[2]["from"] == "gpt" else ""
                corrected = turns[4]["value"] if turns[4]["from"] == "gpt" else ""

                if not question or not flawed or not corrected:
                    continue
                if len(corrected) < 50 or len(flawed) < 50:
                    continue

                pair = make_dpo_pair(
                    system_prompt=system_msg,
                    user_msg=question,
                    chosen_answer=corrected,
                    rejected_answer=flawed,
                    source="self_correction",
                    cancer_type=cancer_type,
                )
                dpo_pairs_all.append(pair)
                type_count += 1

        correction_dpo_count += type_count
        print(f"    {display:25s} → {type_count:>4} DPO pairs")
else:
    print("\n[3/3] SELF-CORRECTION — no correction files found (run section 5e first)")

print(f"  Self-correction DPO pairs: {correction_dpo_count}")


# ======================================================================
# NOTE: Source 4 (Quality Gate — borderline vs random good) was REMOVED.
# It paired borderline answers with random good answers from DIFFERENT
# questions, which is semantically wrong for DPO (the chosen/rejected
# must answer the SAME question). This would teach the model to prefer
# unrelated eloquent text over actual answers.
# ======================================================================


# ── Write all DPO pairs ──────────────────────────────────────────────
random.shuffle(dpo_pairs_all)

with open(DPO_OUTPUT_FILE, "w") as f:
    for pair in dpo_pairs_all:
        f.write(json.dumps(pair) + "\n")

print(f"\n{'='*60}")
print(f"DPO COLLECTION COMPLETE")
print(f"{'='*60}")
print(f"  Grounding rejects:  {grounding_dpo_count:>6,}")
print(f"  Beyond-evidence:    {beyond_dpo_count:>6,}")
print(f"  Self-correction:    {correction_dpo_count:>6,}")
print(f"  {'─'*30}")
print(f"  TOTAL DPO PAIRS:    {len(dpo_pairs_all):>6,}")
print(f"\n  Output: {DPO_OUTPUT_FILE}")
print(f"  Format: TRL chat-template DPO (chosen/rejected message lists)")

# Source distribution
from collections import Counter as DPOCounter
source_dist = DPOCounter(p["source"] for p in dpo_pairs_all)
print(f"\n  Source distribution:")
for src, cnt in source_dist.most_common():
    pct = cnt / max(len(dpo_pairs_all), 1) * 100
    print(f"    {src:<25} {cnt:>5} ({pct:.1f}%)")

del dpo_pairs_all
gc.collect()

## 10a. DPO Dataset Verification

Validates the DPO dataset: checks format consistency, length distributions, source balance, and samples from each source type.

In [ ]:
# ============================================================================
# DPO DATASET VERIFICATION
# ============================================================================

dpo_file = DPO_OUTPUT_FILE
if not os.path.exists(dpo_file):
    print(f"⚠ DPO file not found: {dpo_file}")
    print("  Run the DPO collection cell first.")
else:
    pairs = []
    with open(dpo_file) as f:
        for line in f:
            pairs.append(json.loads(line))

    print(f"DPO DATASET VERIFICATION")
    print(f"{'='*60}")
    print(f"  Total pairs:     {len(pairs):,}")
    print(f"  File:            {dpo_file}")
    print(f"  Size:            {os.path.getsize(dpo_file) / 1024 / 1024:.1f} MB")

    # Format checks
    format_errors = 0
    for i, p in enumerate(pairs):
        if "chosen" not in p or "rejected" not in p:
            format_errors += 1
            continue
        if not isinstance(p["chosen"], list) or not isinstance(p["rejected"], list):
            format_errors += 1
            continue
        if len(p["chosen"]) < 3 or len(p["rejected"]) < 3:
            format_errors += 1

    print(f"  Format errors:   {format_errors}")

    # Source distribution
    sources = Counter(p.get("source", "unknown") for p in pairs)
    print(f"\n  SOURCE DISTRIBUTION")
    print(f"  {'─'*40}")
    for src, cnt in sources.most_common():
        pct = cnt / len(pairs) * 100
        bar = "▪" * max(1, int(pct / 2))
        print(f"    {src:<25} {cnt:>5} ({pct:>5.1f}%) {bar}")

    # Cancer type coverage
    cancer_dist = Counter(p.get("cancer_type", "unknown") for p in pairs)
    print(f"\n  CANCER TYPE COVERAGE")
    print(f"  {'─'*40}")
    for ct, cnt in cancer_dist.most_common():
        display = CANCER_TYPES.get(ct, ct)
        print(f"    {display:<25} {cnt:>5}")

    # Length analysis
    chosen_lens = []
    rejected_lens = []
    for p in pairs:
        chosen_text = p["chosen"][-1]["content"] if p["chosen"] else ""
        rejected_text = p["rejected"][-1]["content"] if p["rejected"] else ""
        chosen_lens.append(len(chosen_text))
        rejected_lens.append(len(rejected_text))

    print(f"\n  LENGTH ANALYSIS")
    print(f"  {'─'*40}")
    print(f"    Chosen answers:   avg={sum(chosen_lens)//max(len(chosen_lens),1):,} chars "
          f"(min={min(chosen_lens, default=0):,}, max={max(chosen_lens, default=0):,})")
    print(f"    Rejected answers: avg={sum(rejected_lens)//max(len(rejected_lens),1):,} chars "
          f"(min={min(rejected_lens, default=0):,}, max={max(rejected_lens, default=0):,})")

    # Thinking block rates
    chosen_think = sum(1 for l in chosen_lens for p in [pairs[chosen_lens.index(l)]]
                       if has_thinking_block(p["chosen"][-1]["content"]))
    # Simpler approach:
    chosen_think = sum(1 for p in pairs if has_thinking_block(p["chosen"][-1].get("content", "")))
    rejected_think = sum(1 for p in pairs if has_thinking_block(p["rejected"][-1].get("content", "")))
    print(f"    Chosen with <think>:   {chosen_think}/{len(pairs)} ({chosen_think/max(len(pairs),1)*100:.1f}%)")
    print(f"    Rejected with <think>: {rejected_think}/{len(pairs)} ({rejected_think/max(len(pairs),1)*100:.1f}%)")

    # Samples
    print(f"\n  SAMPLE PAIRS (one per source)")
    print(f"  {'─'*40}")
    seen_sources = set()
    for p in pairs:
        src = p.get("source", "unknown")
        if src in seen_sources:
            continue
        seen_sources.add(src)

        user_msg = p["chosen"][1]["content"][:200] if len(p["chosen"]) > 1 else "N/A"
        chosen_preview = p["chosen"][-1]["content"][:300] if p["chosen"] else "N/A"
        rejected_preview = p["rejected"][-1]["content"][:300] if p["rejected"] else "N/A"

        print(f"\n    ── {src.upper()} ──")
        print(f"    [USER] {user_msg}...")
        print(f"    [CHOSEN] {chosen_preview}...")
        print(f"    [REJECTED] {rejected_preview}...")

    del pairs
    gc.collect()
    print(f"\n{'='*60}")
    print(f"✓ DPO verification complete")